## CFD


In [ ]:
"""2D lid-driven cavity flow with the streamfunction-vorticity formulation.

Governing equations
    Poisson:              ∇²ψ = -ω
    Vorticity transport:  ∂ω/∂t + u ∂ω/∂x + v ∂ω/∂y = ν ∇²ω
    Velocities:           u = ∂ψ/∂y,   v = -∂ψ/∂x

Boundary conditions
    Top lid:     u = U_lid, v = 0
    Other walls: u = 0,     v = 0
    ψ = 0 on every wall (wall vorticity follows from a 2nd-order finite difference).
"""

from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np


# --------------------------------------------------------------------------- #
# Configuration and results
# --------------------------------------------------------------------------- #
@dataclass(frozen=True)
class CavityConfig:
    nx: int = 128
    ny: int = 128
    Lx: float = 1.0
    Ly: float = 1.0
    Re: float = 100.0
    U_lid: float = 1.0
    nt: int = 20_000
    poisson_iters: int = 500
    log_every: int = 500
    max_dt: float = 1e-3

    @property
    def dx(self) -> float:
        return self.Lx / (self.nx - 1)

    @property
    def dy(self) -> float:
        return self.Ly / (self.ny - 1)

    @property
    def nu(self) -> float:
        return abs(self.U_lid) * self.Lx / self.Re

    @property
    def dt(self) -> float:
        """Time step limited by diffusion, convection and a hard cap."""
        h = min(self.dx, self.dy)
        dt_diff = 0.25 * h**2 / self.nu if self.nu > 0 else self.max_dt
        dt_conv = 0.5 * h / max(abs(self.U_lid), 1e-8)
        return min(dt_diff, dt_conv, self.max_dt)


@dataclass
class CavityResult:
    x: np.ndarray
    y: np.ndarray
    X: np.ndarray
    Y: np.ndarray
    psi: np.ndarray
    omega: np.ndarray
    u: np.ndarray
    v: np.ndarray


# --------------------------------------------------------------------------- #
# Solver
# --------------------------------------------------------------------------- #
class LidDrivenCavitySolver:
    def __init__(self, config: CavityConfig):
        self.cfg = config
        shape = (config.ny, config.nx)
        self.psi = np.zeros(shape)
        self.omega = np.zeros(shape)
        self.u = np.zeros(shape)
        self.v = np.zeros(shape)

    # ---- boundary conditions ------------------------------------------------
    def _apply_velocity_bc(self) -> None:
        u, v = self.u, self.v
        u[0, :] = u[:, 0] = u[:, -1] = 0.0
        u[-1, :] = self.cfg.U_lid
        v[0, :] = v[-1, :] = v[:, 0] = v[:, -1] = 0.0

    def _apply_vorticity_bc(self) -> None:
        """Wall vorticity from ψ = 0 on the walls (2nd-order finite differences)."""
        c, w, psi = self.cfg, self.omega, self.psi
        dx2, dy2 = c.dx**2, c.dy**2

        w[0, 1:-1] = -2.0 * psi[1, 1:-1] / dy2
        w[-1, 1:-1] = -2.0 * psi[-2, 1:-1] / dy2 - 2.0 * c.U_lid / c.dy
        w[1:-1, 0] = -2.0 * psi[1:-1, 1] / dx2
        w[1:-1, -1] = -2.0 * psi[1:-1, -2] / dx2

        # Corners: average of the adjacent wall values
        w[0, 0] = 0.5 * (w[0, 1] + w[1, 0])
        w[0, -1] = 0.5 * (w[0, -2] + w[1, -1])
        w[-1, 0] = 0.5 * (w[-1, 1] + w[-2, 0])
        w[-1, -1] = 0.5 * (w[-1, -2] + w[-2, -1])

    # ---- field updates --------------------------------------------------------
    def _solve_poisson(self) -> None:
        """Jacobi iterations for ∇²ψ = -ω with ψ = 0 on the walls."""
        c, psi, w = self.cfg, self.psi, self.omega
        dx2, dy2 = c.dx**2, c.dy**2
        beta = dx2 * dy2 / (2.0 * (dx2 + dy2))

        for _ in range(c.poisson_iters):
            psi[1:-1, 1:-1] = beta * (
                (psi[1:-1, 2:] + psi[1:-1, :-2]) / dx2
                + (psi[2:, 1:-1] + psi[:-2, 1:-1]) / dy2
                + w[1:-1, 1:-1]
            )
            # Walls stay at ψ = 0 (they are never modified above, so this is
            # only a safeguard)
            psi[[0, -1], :] = 0.0
            psi[:, [0, -1]] = 0.0

    def _update_velocity(self) -> None:
        c, psi = self.cfg, self.psi
        self.u[1:-1, 1:-1] = (psi[2:, 1:-1] - psi[:-2, 1:-1]) / (2.0 * c.dy)
        self.v[1:-1, 1:-1] = -(psi[1:-1, 2:] - psi[1:-1, :-2]) / (2.0 * c.dx)
        self._apply_velocity_bc()

    def _advance_vorticity(self, dt: float) -> None:
        """One explicit Euler step of the vorticity transport equation."""
        c, w = self.cfg, self.omega
        w_old = w.copy()
        inner = w_old[1:-1, 1:-1]

        dwdx = (w_old[1:-1, 2:] - w_old[1:-1, :-2]) / (2.0 * c.dx)
        dwdy = (w_old[2:, 1:-1] - w_old[:-2, 1:-1]) / (2.0 * c.dy)
        d2wdx2 = (w_old[1:-1, 2:] - 2.0 * inner + w_old[1:-1, :-2]) / c.dx**2
        d2wdy2 = (w_old[2:, 1:-1] - 2.0 * inner + w_old[:-2, 1:-1]) / c.dy**2

        convection = self.u[1:-1, 1:-1] * dwdx + self.v[1:-1, 1:-1] * dwdy
        diffusion = c.nu * (d2wdx2 + d2wdy2)
        w[1:-1, 1:-1] = inner + dt * (diffusion - convection)

        self._apply_vorticity_bc()
        return w_old

    # ---- main loop ------------------------------------------------------------
    def run(self) -> CavityResult:
        c = self.cfg
        dt = c.dt

        self._apply_velocity_bc()
        self._apply_vorticity_bc()

        for n in range(c.nt):
            self._solve_poisson()
            self._update_velocity()
            self._apply_vorticity_bc()
            w_old = self._advance_vorticity(dt)

            if n % c.log_every == 0:
                max_change = np.max(np.abs(self.omega - w_old))
                print(f"Step {n:6d} | max(|Δω|) = {max_change:.3e}")

        # Final consistent ψ and velocity fields
        self._solve_poisson()
        self._update_velocity()

        x = np.linspace(0.0, c.Lx, c.nx)
        y = np.linspace(0.0, c.Ly, c.ny)
        X, Y = np.meshgrid(x, y)
        return CavityResult(x, y, X, Y, self.psi, self.omega, self.u, self.v)


# --------------------------------------------------------------------------- #
# Plotting
# --------------------------------------------------------------------------- #
def plot_streamlines(r: CavityResult) -> None:
    plt.figure(figsize=(7, 6))
    plt.streamplot(r.X, r.Y, r.u, r.v, density=2.0, linewidth=1.0, arrowsize=1.0)
    plt.title("Streamlines - Lid Driven Cavity")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.xlim(r.x[0], r.x[-1])
    plt.ylim(r.y[0], r.y[-1])
    plt.gca().set_aspect("equal")
    plt.tight_layout()


def plot_centerline(x, u_line, *, xlabel, ylabel, title, label) -> None:
    plt.figure(figsize=(6, 5))
    plt.plot(x, u_line, label=label)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()


def plot_results(r: CavityResult) -> None:
    ix, iy = len(r.x) // 2, len(r.y) // 2

    plot_streamlines(r)
    plot_centerline(
        r.u[:, ix], r.y,
        xlabel="u", ylabel="y",
        title="u along the central vertical line", label="u(x=0.5, y)",
    )
    plot_centerline(
        r.x, r.u[iy, :],
        xlabel="x", ylabel="u",
        title="u along the central horizontal line", label="u(x, y=0.5)",
    )
    plt.show()


# --------------------------------------------------------------------------- #
# Entry point
# --------------------------------------------------------------------------- #
if __name__ == "__main__":
    config = CavityConfig(nx=129, ny=129, Re=100.0, U_lid=-1.0, nt=50_000)
    result = LidDrivenCavitySolver(config).run()
    plot_results(result)

## PINN - Navier-Stokes


In [ ]:
import torch
import torch.nn as nn
from torch.quasirandom import SobolEngine
import numpy as np
import matplotlib.colors as mcolors

torch.manual_seed(88)
np.random.seed(88)

if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")

# ============================================================
# 2) Amostrador de dados esparsos
# ============================================================

def sample_sparse_data_from_reference(x, y, u, v, p=None,
                                       n_data=1000, seed=42):
    """
    Amostra dados esparsos de u, v (e opcionalmente p) a partir da CFD de referência.
    A pressão p costuma ser opcional em PINNs (Hidden Fluid Mechanics).
    """
    rng = np.random.default_rng(seed)
    Xg, Yg = np.meshgrid(x, y, indexing="xy")

    # Exclui os pontos de contorno da amostragem
    mask = np.ones_like(Xg, dtype=bool)
    mask[0, :] = mask[-1, :] = mask[:, 0] = mask[:, -1] = False

    if p is not None:
        data = np.column_stack([Xg[mask], Yg[mask], u[mask], v[mask], p[mask]])
    else:
        data = np.column_stack([Xg[mask], Yg[mask], u[mask], v[mask]])

    idx  = rng.choice(data.shape[0], size=n_data, replace=False)
    data = data[idx]

    batch = {
        "xy": data[:, 0:2],
        "u":  data[:, 2:3],
        "v":  data[:, 3:4],
    }
    if p is not None:
        batch["p"] = data[:, 4:5]

    return batch


# ============================================================
# Embedding de Fourier aleatório (Random Fourier Features)
# ============================================================

class FourierEmbedding(nn.Module):
    """
    Mapeia (x, y) → [sin(Bx), cos(Bx)], com B ~ N(0, σ²).
    Permite que a rede represente variações espaciais de alta frequência.
    """
    def __init__(self, in_dim: int = 2, n_freq: int = 32, sigma: float = 2.0):
        super().__init__()
        B = torch.randn(in_dim, n_freq) * sigma
        self.register_buffer("B", B)          # buffer fixo, não é parâmetro treinável

    @property
    def out_dim(self):
        return 2 * self.B.shape[1]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        proj = x @ self.B                     # (N, n_freq)
        return torch.cat([torch.sin(proj),
                          torch.cos(proj)], dim=-1)   # (N, 2*n_freq)


def build_mlp(layer_sizes):
    modules = []
    for i in range(len(layer_sizes) - 2):
        modules += [nn.Linear(layer_sizes[i], layer_sizes[i + 1]), nn.Tanh()]
    modules.append(nn.Linear(layer_sizes[-2], layer_sizes[-1]))
    return nn.Sequential(*modules)


# ============================================================
# CORREÇÃO 2 – Rede mais profunda + entrada de Fourier + normalização da entrada
# ============================================================

class CavityPINN_NS(nn.Module):
    """
    Retorna (u, v, p).
    """
    def __init__(
        self,
        hidden_layers = (128, 128, 128, 128),
        n_freq: int   = 32,
        sigma: float  = 2.0,
        Lx: float     = 1.0,
        Ly: float     = 1.0,
    ):
        super().__init__()
        # Buffers de normalização da entrada
        self.register_buffer("x_mean", torch.tensor([Lx / 2.0, Ly / 2.0]))
        self.register_buffer("x_std",  torch.tensor([Lx / 2.0, Ly / 2.0]))

        # Embedding de Fourier
        self.fourier = FourierEmbedding(2, n_freq, sigma)
        in_dim = self.fourier.out_dim

        # A saída agora tem 3 componentes: u, v, p
        layer_sizes = [in_dim] + list(hidden_layers) + [3]
        self.net = build_mlp(layer_sizes)

    def forward(self, xy: torch.Tensor):
        xy_norm = (xy - self.x_mean) / self.x_std    # [-1, 1]
        feats   = self.fourier(xy_norm)
        out     = self.net(feats)
        return out[:, 0:1], out[:, 1:2], out[:, 2:3] # u, v, p


# ============================================================
# CORREÇÃO 3 – Cálculo eficiente de gradientes (Navier-Stokes)
# ============================================================

def physics_residuals(model, xy, nu):
    """
    Continuidade:      u_x + v_y = 0
    Momento em x:      u*u_x + v*u_y + p_x - ν*(u_xx + u_yy) = 0
    Momento em y:      u*v_x + v*v_y + p_y - ν*(v_xx + v_yy) = 0
    """
    # 1. Passagem direta (forward)
    u, v, p = model(xy)

    # 2. Derivadas de primeira ordem (calculadas de uma só vez para cada saída,
    #    economizando passagens reversas)
    u_grad = torch.autograd.grad(u, xy, grad_outputs=torch.ones_like(u), create_graph=True)[0]
    u_x, u_y = u_grad[:, 0:1], u_grad[:, 1:2]

    v_grad = torch.autograd.grad(v, xy, grad_outputs=torch.ones_like(v), create_graph=True)[0]
    v_x, v_y = v_grad[:, 0:1], v_grad[:, 1:2]

    p_grad = torch.autograd.grad(p, xy, grad_outputs=torch.ones_like(p), create_graph=True)[0]
    p_x, p_y = p_grad[:, 0:1], p_grad[:, 1:2]

    # 3. Derivadas de segunda ordem (só as diagonais: u_xx, u_yy, v_xx, v_yy)
    u_x_grad = torch.autograd.grad(u_x, xy, grad_outputs=torch.ones_like(u_x), create_graph=True)[0]
    u_xx = u_x_grad[:, 0:1]

    u_y_grad = torch.autograd.grad(u_y, xy, grad_outputs=torch.ones_like(u_y), create_graph=True)[0]
    u_yy = u_y_grad[:, 1:2]

    v_x_grad = torch.autograd.grad(v_x, xy, grad_outputs=torch.ones_like(v_x), create_graph=True)[0]
    v_xx = v_x_grad[:, 0:1]

    v_y_grad = torch.autograd.grad(v_y, xy, grad_outputs=torch.ones_like(v_y), create_graph=True)[0]
    v_yy = v_y_grad[:, 1:2]

    # 4. Resíduos de Navier-Stokes
    r_cont  = u_x + v_y
    r_mom_x = u * u_x + v * u_y + p_x - nu * (u_xx + u_yy)
    r_mom_y = u * v_x + v * v_y + p_y - nu * (v_xx + v_yy)

    return r_cont, r_mom_x, r_mom_y


# ============================================================
# 3) Geradores de pontos de colocação e de contorno
# ============================================================

def make_sobol_points(n_points, Lx, Ly, device, seed=123):
    sobol = SobolEngine(dimension=2, scramble=True, seed=seed)
    pts   = sobol.draw(n_points).to(device)
    pts[:, 0] *= Lx
    pts[:, 1] *= Ly
    return pts


def make_boundary_points(n_per_side, Lx, Ly, U_lid, device, seed=123):
    rng = np.random.default_rng(seed)

    xb = rng.random((n_per_side, 1)) * Lx;  yb = np.zeros((n_per_side, 1))
    xt = rng.random((n_per_side, 1)) * Lx;  yt = np.full((n_per_side, 1), Ly)
    xl = np.zeros((n_per_side, 1));          yl = rng.random((n_per_side, 1)) * Ly
    xr = np.full((n_per_side, 1), Lx);      yr = rng.random((n_per_side, 1)) * Ly

    xy_bc = np.vstack([
        np.hstack([xb, yb]), np.hstack([xt, yt]),
        np.hstack([xl, yl]), np.hstack([xr, yr]),
    ])
    u_bc = np.vstack([
        np.zeros((n_per_side, 1)),
        np.full((n_per_side, 1), U_lid),
        np.zeros((n_per_side, 1)),
        np.zeros((n_per_side, 1)),
    ])

    return (
        torch.tensor(xy_bc, dtype=torch.float32, device=device),
        torch.tensor(u_bc,  dtype=torch.float32, device=device),
    )

def to_device_tensor(arr, device):
    return torch.tensor(arr, dtype=torch.float32, device=device)


# ============================================================
# 4) Termos individuais da função de perda
# ============================================================

def loss_data(model, xy_d, u_d, v_d, p_d=None):
    xy_d = xy_d.detach().requires_grad_(True)
    u_p, v_p, p_p = model(xy_d)

    loss = nn.functional.mse_loss(u_p, u_d) + nn.functional.mse_loss(v_p, v_d)
    if p_d is not None:
        loss += 0.05 * nn.functional.mse_loss(p_p, p_d)
    return loss


def loss_physics(model, xy_f, nu):
    xy_f = xy_f.detach().requires_grad_(True)
    r_c, r_mx, r_my = physics_residuals(model, xy_f, nu)
    return torch.mean(r_c ** 2) + torch.mean(r_mx ** 2) + torch.mean(r_my ** 2)


def loss_boundary(model, xy_b, u_bc):
    """
    Imposição explícita de não-deslizamento em cada parede:
      - Inferior (y=0) : u=0, v=0
      - Esquerda (x=0) : u=0, v=0
      - Direita  (x=1) : u=0, v=0
      - Superior (y=1) : u=U_lid, v=0  (tampa móvel)
    """
    n = xy_b.shape[0] // 4   # pontos por parede

    xy_bottom = xy_b[0*n : 1*n].detach().requires_grad_(True)
    xy_top    = xy_b[1*n : 2*n].detach().requires_grad_(True)
    xy_left   = xy_b[2*n : 3*n].detach().requires_grad_(True)
    xy_right  = xy_b[3*n : 4*n].detach().requires_grad_(True)

    u_lid_target = u_bc[1*n : 2*n]   # somente o topo tem U_lid != 0

    # Predição dos campos
    u_bot, v_bot, _ = model(xy_bottom)
    u_top, v_top, _ = model(xy_top)
    u_lft, v_lft, _ = model(xy_left)
    u_rgt, v_rgt, _ = model(xy_right)

    # Paredes com não-deslizamento: u=0, v=0
    loss_no_slip = (
          torch.mean(u_bot**2) + torch.mean(v_bot**2)
        + torch.mean(u_lft**2) + torch.mean(v_lft**2)
        + torch.mean(u_rgt**2) + torch.mean(v_rgt**2)
    )

    # Tampa móvel: u=U_lid, v=0
    loss_lid = (
          torch.mean((u_top - u_lid_target)**2)
        + torch.mean(v_top**2)
    )

    return loss_no_slip + loss_lid

# ============================================================
# CORREÇÕES 4 e 5 – Peso da física crescente + treino em duas fases (Adam → L-BFGS)
# ============================================================

def train_pinn(
    model,
    xy_data_all, u_data_all, v_data_all, p_data_all,
    xy_f_all, xy_b_all, u_bc_all,
    nu,
    adam_epochs  = 3000,
    batch_data   = 200,
    batch_f      = 2048,
    batch_b      = 512,
    lr           = 1e-3,
    w_data       = 5.0,
    w_phys_start = 1.0,
    w_phys_end   = 10.0,
    w_bc         = 10.0,
    lbfgs_steps  = 60,
    lbfgs_max_iter = 200,
):
    device = xy_f_all.device
    Nd, Nf, Nb = xy_data_all.shape[0], xy_f_all.shape[0], xy_b_all.shape[0]

    # ---- Fase Adam ----
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.MultiStepLR(
        optimizer, milestones=[1000, 2000], gamma=0.3
    )

    history = {"total": [], "data": [], "physics": [], "bc": []}

    for epoch in range(adam_epochs):
        idx_d = torch.randint(0, Nd, (min(batch_data, Nd),), device=device)
        idx_f = torch.randint(0, Nf, (min(batch_f,  Nf),),  device=device)

        t = epoch / max(adam_epochs - 1, 1)
        w_phys = w_phys_start + t * (w_phys_end - w_phys_start)

        optimizer.zero_grad()

        # Trata a pressão opcional
        p_batch = p_data_all[idx_d] if p_data_all is not None else None

        ld = loss_data(
            model,
            xy_data_all[idx_d], u_data_all[idx_d], v_data_all[idx_d], p_batch
        )
        lp = loss_physics(model, xy_f_all[idx_f], nu)

        # Amostra por parede para preservar a ordenação:
        n_wall = Nb // 4
        b_per_wall = min(batch_b // 4, n_wall)
        idx_b = torch.cat([
            torch.randint(i*n_wall, (i+1)*n_wall, (b_per_wall,), device=device)
            for i in range(4)
        ])
        lb = loss_boundary(model, xy_b_all[idx_b], u_bc_all[idx_b])

        loss = w_data * ld + w_phys * lp + w_bc * lb
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        history["total"].append(loss.item())
        history["data"].append(ld.item())
        history["physics"].append(lp.item())
        history["bc"].append(lb.item())

        if epoch % 300 == 0:
            print(
                f"  Adam {epoch:5d}/{adam_epochs} | "
                f"Total={loss.item():.3e} | "
                f"Dados={ld.item():.3e} | "
                f"Física={lp.item():.3e} (w={w_phys:.1f}) | "
                f"CC={lb.item():.3e}"
            )

    # ---- CORREÇÃO 5: fase L-BFGS (batch completo) ----
    print("\nIniciando fase L-BFGS...")
    optimizer_lbfgs = torch.optim.LBFGS(
        model.parameters(),
        lr=0.1,
        max_iter=lbfgs_max_iter,
        tolerance_grad=1e-9,
        tolerance_change=1e-12,
        history_size=100,
        line_search_fn="strong_wolfe",
    )

    lbfgs_history = {"total": [], "data": [], "physics": [], "bc": []}

    def closure():
        optimizer_lbfgs.zero_grad()
        ld = loss_data(
            model,
            xy_data_all, u_data_all, v_data_all, p_data_all
        )
        lp = loss_physics(model, xy_f_all, nu)
        lb = loss_boundary(model, xy_b_all, u_bc_all)
        total = w_data * ld + w_phys_end * lp + w_bc * lb
        total.backward()
        lbfgs_history["total"].append(total.item())
        lbfgs_history["data"].append(ld.item())
        lbfgs_history["physics"].append(lp.item())
        lbfgs_history["bc"].append(lb.item())
        return total

    for step in range(lbfgs_steps):
        optimizer_lbfgs.step(closure)
        if len(lbfgs_history["total"]) > 0:
            print(
                f"  L-BFGS passo {step + 1:3d}/{lbfgs_steps} | "
                f"Total={lbfgs_history['total'][-1]:.3e} | "
                f"Dados={lbfgs_history['data'][-1]:.3e} | "
                f"Física={lbfgs_history['physics'][-1]:.3e} | "
                f"CC={lbfgs_history['bc'][-1]:.3e}"
            )

    for k in history:
        history[k] += lbfgs_history[k]

    return history


# ============================================================
# 5) Avaliação na malha
# ============================================================

def evaluate_pinn_on_grid(model, x, y, device):
    Xg, Yg = np.meshgrid(x, y, indexing="xy")
    xy = np.column_stack([Xg.ravel(), Yg.ravel()])
    xy_t = torch.tensor(xy, dtype=torch.float32, device=device).requires_grad_(True)

    with torch.no_grad():
        u_p, v_p, p_p = model(xy_t)

    def to_grid(t):
        return t.detach().cpu().numpy().reshape(Xg.shape)

    return Xg, Yg, to_grid(u_p), to_grid(v_p), to_grid(p_p)

# ============================================================
# Ponto de entrada da execução
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {device}\n")

# OBS.: Garanta que as variáveis `x, y, u, v` existam no escopo global
# (carregadas a partir da sua solução CFD). Forneça `p` caso tenha dados de
# pressão; caso contrário, ele será None.
try:
    _p_ref = p  # se p existir
except NameError:
    _p_ref = None

# ----------------------------------------------------------
# Dados esparsos da CFD
# ----------------------------------------------------------
sparse = sample_sparse_data_from_reference(
    x=x, y=y,
    u=u, v=v, p=_p_ref,
    n_data=3600, seed=8,
)

xy_data_all    = to_device_tensor(sparse["xy"], device)
u_data_all     = to_device_tensor(sparse["u"],  device)
v_data_all     = to_device_tensor(sparse["v"],  device)
p_data_all     = to_device_tensor(sparse["p"],  device) if "p" in sparse else None

# ----------------------------------------------------------
# Pontos de colocação e de contorno
Lx, Ly  = 1.0, 1.0
Re = 1000
U_lid = -1
nu = abs(U_lid) * Lx / Re
# ----------------------------------------------------------
xy_f_all         = make_sobol_points(2048, Lx, Ly, device, seed=188)
xy_b_all, u_bc_all = make_boundary_points(256, Lx, Ly, U_lid, device, seed=88)

# ----------------------------------------------------------
# Modelo
# ----------------------------------------------------------
model = CavityPINN_NS(
    hidden_layers=(128, 128, 128, 128),
    n_freq=48,
    sigma=4.0,
    Lx=Lx, Ly=Ly,
).to(device)

total_params = sum(param.numel() for param in model.parameters() if param.requires_grad)
print(f"Parâmetros da PINN (Navier-Stokes): {total_params:,}\n")

# ----------------------------------------------------------
# Treinamento
# ----------------------------------------------------------
print("Treinando a PINN...")
history = train_pinn(
    model=model,
    xy_data_all=xy_data_all,
    u_data_all=u_data_all,
    v_data_all=v_data_all,
    p_data_all=p_data_all,
    xy_f_all=xy_f_all,
    xy_b_all=xy_b_all,
    u_bc_all=u_bc_all,
    nu=nu,
    adam_epochs=5100,
    batch_data=200,
    batch_f=512,
    batch_b=512,
    lr=1e-3,
    w_data=40.0,
    w_phys_start=1.0,
    w_phys_end=40.0,
    w_bc=40.0,
    lbfgs_steps=80,
    lbfgs_max_iter=200,
)
print("Treinamento concluído.\n")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# ============================================================
# Main table-generation and plotting function
# ============================================================

def plot_centerlines(x, y, u_ref, v_ref, u_pinn, v_pinn):
    """
    Generate plots comparing u and v velocities on the centerlines.
    """
    ix_center = len(x) // 2 
    iy_center = len(y) // 2   

    fig, ax = plt.subplots(1, 2, figsize=(12, 5))

    # Plot 1: u velocity along the vertical centerline
    ax[0].plot(u_ref[:, ix_center], y, 'k-', linewidth=2, label='CFD Reference')
    ax[0].plot(u_pinn[:, ix_center], y, 'r--', linewidth=2, label='PINN')
    ax[0].set_xlabel('u - velocity')
    ax[0].set_ylabel('y')
    ax[0].set_title('u velocity along vertical centerline (x ≈ 0.5)')
    ax[0].grid(True, linestyle=':', alpha=0.7)
    ax[0].legend()

    # Plot 2: v velocity along the horizontal centerline
    ax[1].plot(x, v_ref[iy_center, :], 'k-', linewidth=2, label='CFD Reference')
    ax[1].plot(x, v_pinn[iy_center, :], 'r--', linewidth=2, label='PINN')
    ax[1].set_xlabel('x')
    ax[1].set_ylabel('v - velocity')
    ax[1].set_title('v velocity along horizontal centerline (y ≈ 0.5)')
    ax[1].grid(True, linestyle=':', alpha=0.7)
    ax[1].legend()

    plt.tight_layout()
    plt.show()

def save_analysis_tables(
    x, y,
    u_ref,  v_ref,
    u_pinn, v_pinn, p_pinn,
    output_dir=".",
):
    """
    Generate and save centerline comparison tables as CSV files.
    """
    os.makedirs(output_dir, exist_ok=True)

    ix_center = len(x) // 2  
    iy_center = len(y) // 2  


    p_pinn_centered = p_pinn - p_pinn.mean()

    # ===========================================================
    # Table 1 – Horizontal velocity & pressure along vertical centreline (x = 0.5)
    # ===========================================================
    df1 = pd.DataFrame({
        "y":          y,
        "u (CFD)":    u_ref[:, ix_center],
        "u (PINN)":   u_pinn[:, ix_center],
        "p (PINN)":   p_pinn_centered[:, ix_center],
    })
    
    path1 = os.path.join(output_dir, "table1_vertical_centerline.csv")
    df1.to_csv(path1, index=False)
    print(f"--- Vertical centreline (x ≈ 0.5) saved to {path1} ---")
    print(df1.to_string(index=False), "\n")

    # ===========================================================
    # Table 2 – Vertical velocity & pressure along horizontal centreline (y = 0.5)
    # ===========================================================
    df2 = pd.DataFrame({
        "x":          x,
        "v (CFD)":    v_ref[iy_center, :],
        "v (PINN)":   v_pinn[iy_center, :],
        "p (PINN)":   p_pinn_centered[iy_center, :],
    })
    
    path2 = os.path.join(output_dir, "table2_horizontal_centerline.csv")
    df2.to_csv(path2, index=False)
    print(f"--- Horizontal centreline (y ≈ 0.5) saved to {path2} ---")
    print(df2.to_string(index=False), "\n")

    return df1, df2

# ============================================================
# Execution Entry Point
# ============================================================

print("Evaluating PINN on grid...")
Xg, Yg, u_pinn, v_pinn, p_pinn = evaluate_pinn_on_grid(model, x, y, device)

print("Generating plots...")
plot_centerlines(x, y, u, v, u_pinn, v_pinn)

print("Generating tables...")
df_vert, df_horz = save_analysis_tables(
    x=x, y=y,
    u_ref=u,       v_ref=v,
    u_pinn=u_pinn, v_pinn=v_pinn, p_pinn=p_pinn,
    output_dir="."
)

def save_analysis_tables(
    x, y,
    u_ref, v_ref,
    u_pinn, v_pinn, p_pinn,
    output_dir=".",
):
    import os
    os.makedirs(output_dir, exist_ok=True)

    dx = x[1] - x[0]
    dy = y[1] - y[0]

    ix_center = len(x) // 2
    iy_center = len(y) // 2

    # ===========================================================
    # Table 1 – Velocity magnitude peak (replaces vortex/psi stats)
    # ===========================================================
    def peak_speed_stats(u, v, x, y):
        speed = np.sqrt(u**2 + v**2)
        idx   = np.unravel_index(np.argmax(speed), speed.shape)
        iy, ix = idx
        return {
            "|U| (peak)": float(speed[iy, ix]),
            "u":          float(u[iy, ix]),
            "v":          float(v[iy, ix]),
            "x":          float(x[ix]),
            "y":          float(y[iy]),
        }

    df1 = pd.DataFrame(
        [peak_speed_stats(u_ref, v_ref, x, y),
         peak_speed_stats(u_pinn, v_pinn, x, y)],
        index=["CFD", "PINN"],
    )
    df1.index.name = "Reference"
    print(df1.to_string(), "\n")

    # ===========================================================
    # Table 2 – u and p along vertical centreline (x ≈ 0.5)
    # ===========================================================
    df2 = pd.DataFrame({
        "y":        y,
        "u (CFD)":  u_ref[:, ix_center],
        "u (PINN)": u_pinn[:, ix_center],
        "p (PINN)": p_pinn[:, ix_center],
    })
    print(df2.to_string(index=False), "\n")

    # ===========================================================
    # Table 3 – v and p along horizontal centreline (y ≈ 0.5)
    # ===========================================================
    df3 = pd.DataFrame({
        "x":        x,
        "v (CFD)":  v_ref[iy_center, :],
        "v (PINN)": v_pinn[iy_center, :],
        "p (PINN)": p_pinn[iy_center, :],
    })
    print(df3.to_string(index=False), "\n")

    # ===========================================================
    # Table 4 – Pressure field stats
    # ===========================================================
    df4 = pd.DataFrame({
        "Metric": ["min p", "max p", "mean p", "p at centre"],
        "PINN":   [
            float(p_pinn.min()),
            float(p_pinn.max()),
            float(p_pinn.mean()),
            float(p_pinn[iy_center, ix_center]),
        ],
    })
    print(df4.to_string(index=False), "\n")

    # ===========================================================
    # Table 5 – u-velocity on the moving lid (y = 1)
    # ===========================================================
    df5 = pd.DataFrame({
        "x":           x,
        "u (CFD)":     u_ref[-1, :],
        "u (PINN)":    u_pinn[-1, :],
    })
    print(df5.to_string(index=False), "\n")

    return df1, df2, df3, df4, df5

# ---- Unpack evaluate_pinn_on_grid (NS primitive-variable version) ----
Xg, Yg, u_pinn, v_pinn, p_pinn = evaluate_pinn_on_grid(model, x, y, device)

ix_center = len(x) // 2
iy_center = len(y) // 2

u_ref = u
v_ref = v

# ---- Plot 1: Streamlines side-by-side ----
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, u_plot, v_plot, title in zip(
    axes,
    [u_ref, u_pinn],
    [v_ref, v_pinn],
    ["Reference CFD", "PINN (NS)"],
):
    ax.streamplot(Xg, Yg, u_plot, v_plot, density=2.0, linewidth=0.9, arrowsize=0.9)
    ax.set_title(title)
    ax.set_xlabel("x"); ax.set_ylabel("y")
    ax.set_xlim(0, 1);  ax.set_ylim(0, 1)
    ax.set_aspect("equal")
plt.tight_layout()
plt.show()

# ---- Plot 2: u along central vertical line (x = 0.5) ----
plt.figure(figsize=(6, 5))
plt.plot(u_ref[:, ix_center],  y, "k-",  linewidth=2.0, label="Reference CFD")
plt.plot(u_pinn[:, ix_center], y, "r--", linewidth=2.0, label="PINN (NS)")
plt.title("u-velocity — central vertical line (x = 0.5)")
plt.xlabel("u"); plt.ylabel("y")
plt.grid(True, alpha=0.3); plt.legend()
plt.tight_layout()
plt.show()

# ---- Plot 3: u along central horizontal line (y = 0.5) ----
plt.figure(figsize=(6, 5))
plt.plot(x, u_ref[iy_center, :],  "k-",  linewidth=2.0, label="Reference CFD")
plt.plot(x, u_pinn[iy_center, :], "r--", linewidth=2.0, label="PINN (NS)")
plt.title("u-velocity — central horizontal line (y = 0.5)")
plt.xlabel("x"); plt.ylabel("u")
plt.grid(True, alpha=0.3); plt.legend()
plt.tight_layout()
plt.show()

# ---- Plot 4: v along central vertical line (x = 0.5) ----
plt.figure(figsize=(6, 5))
plt.plot(v_ref[:, ix_center],  y, "k-",  linewidth=2.0, label="Reference CFD")
plt.plot(v_pinn[:, ix_center], y, "r--", linewidth=2.0, label="PINN (NS)")
plt.title("v-velocity — central vertical line (x = 0.5)")
plt.xlabel("v"); plt.ylabel("y")
plt.grid(True, alpha=0.3); plt.legend()
plt.tight_layout()
plt.show()

# ---- Plot 5: Pressure field (PINN only — no CFD p reference needed) ----
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.pcolormesh(Xg, Yg, p_pinn, cmap="RdBu_r", shading="gouraud")
fig.colorbar(im, ax=ax, label="p")
ax.set_title("PINN pressure field")
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.set_xlim(0, 1);  ax.set_ylim(0, 1)
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

# ---- Plot 6: Loss history ----
plt.figure(figsize=(8, 5))
plt.semilogy(history["total"],   label="Total",   linewidth=1.5)
plt.semilogy(history["data"],    label="Data",    linewidth=1.5)
plt.semilogy(history["physics"], label="Physics", linewidth=1.5)
plt.semilogy(history["bc"],      label="BC",      linewidth=1.5)
plt.semilogy(history["gauge"],   label="Gauge",   linewidth=1.5)
plt.axvline(adam_epochs, color="gray", linestyle=":", linewidth=1.0, label="Adam → L-BFGS")
plt.title("Training loss history")
plt.xlabel("Iteration"); plt.ylabel("Loss (log scale)")
plt.grid(True, alpha=0.3); plt.legend()
plt.tight_layout()
plt.show()

def rel_l2(pred, ref):
    return np.linalg.norm(pred - ref) / (np.linalg.norm(ref) + 1e-12)

print("\n--- Relative L2 errors on full grid ---")
print(f"  u : {rel_l2(u_pinn, u_ref) * 100:.2f} %")
print(f"  v : {rel_l2(v_pinn, v_ref) * 100:.2f} %")


print("\n--- Statistical verification of U_lid and Re from PINN fields ---")

u_top_pinn = u_pinn[-1, :]
u_top_ref  = u_ref[-1, :]

U_lid_pinn = float(np.mean(u_top_pinn))
U_lid_ref  = float(np.mean(u_top_ref))
U_lid_std  = float(np.std(u_top_pinn))

print(f"\n  U_lid (original)       : {U_lid:.4f}")
print(f"  U_lid from CFD ref     : {U_lid_ref:.4f}")
print(f"  U_lid from PINN fields : {U_lid_pinn:.4f}  (std = {U_lid_std:.4f})")
print(f"  Relative error vs CFD  : {abs(U_lid_pinn - U_lid_ref) / abs(U_lid_ref) * 100:.2f} %")


dx_g = x[1] - x[0]
dy_g = y[1] - y[0]

u_i = u_pinn[1:-1, 1:-1]
v_i = v_pinn[1:-1, 1:-1]

du_dx = (u_pinn[1:-1, 2:] - u_pinn[1:-1, :-2]) / (2 * dx_g)
du_dy = (u_pinn[2:, 1:-1] - u_pinn[:-2, 1:-1]) / (2 * dy_g)
dp_dx = (p_pinn[1:-1, 2:] - p_pinn[1:-1, :-2]) / (2 * dx_g)


lap_u = (
    (u_pinn[1:-1, 2:] - 2*u_pinn[1:-1, 1:-1] + u_pinn[1:-1, :-2]) / dx_g**2
  + (u_pinn[2:, 1:-1] - 2*u_pinn[1:-1, 1:-1] + u_pinn[:-2, 1:-1]) / dy_g**2
)


advection_x = (u_i * du_dx + v_i * du_dy + dp_dx).ravel()
diffusion_x = lap_u.ravel()

mask_valid = np.abs(diffusion_x) > 1e-6
nu_fit = (np.dot(diffusion_x[mask_valid], advection_x[mask_valid])
          / np.dot(diffusion_x[mask_valid], diffusion_x[mask_valid]))

Re_pinn = abs(U_lid_pinn) * Lx / nu_fit

print(f"\n  Re (original)          : {Re:.2f}")
print(f"  nu fitted from PINN    : {nu_fit:.5f}  (original nu = {nu:.5f})")
print(f"  Re from PINN fields    : {Re_pinn:.2f}")
print(f"  Relative error vs CFD  : {abs(Re_pinn - Re) / Re * 100:.2f} %")

# ---- U_lid verification plot ----
plt.figure(figsize=(7, 4))
plt.plot(x, u_top_ref,  "k-",  linewidth=2.0, label=f"CFD ref  (mean={U_lid_ref:.4f})")
plt.plot(x, u_top_pinn, "r--", linewidth=2.0, label=f"PINN     (mean={U_lid_pinn:.4f})")
plt.axhline(U_lid, color="gray", linestyle=":", linewidth=1.5, label=f"Target U_lid={U_lid}")
plt.title("u-velocity at top wall (y = 1.0) — U_lid verification")
plt.xlabel("x"); plt.ylabel("u")
plt.grid(True, alpha=0.3); plt.legend()
plt.tight_layout()
plt.show()


speed_ref  = np.sqrt(u_ref**2  + v_ref**2)
speed_pinn = np.sqrt(u_pinn**2 + v_pinn**2)

vmax = max(speed_ref.max(), speed_pinn.max())
norm = mcolors.Normalize(vmin=0.0, vmax=vmax)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, speed, title in zip(
    axes,
    [speed_ref, speed_pinn],
    ["Reference CFD", "PINN (NS)"],
):
    im = ax.pcolormesh(Xg, Yg, speed, cmap="plasma", norm=norm, shading="gouraud")
    ax.set_title(title)
    ax.set_xlabel("x"); ax.set_ylabel("y")
    ax.set_xlim(0, 1);  ax.set_ylim(0, 1)
    ax.set_aspect("equal")
fig.colorbar(im, ax=axes[1], label="|U| velocity modulus", fraction=0.05)
plt.suptitle("Velocity modulus", y=1.02)
plt.tight_layout()
plt.show()


save_analysis_tables(
    x=x, y=y,
    u_ref=u_ref, v_ref=v_ref,
    u_pinn=u_pinn, v_pinn=v_pinn, p_pinn=p_pinn,
    output_dir=".",
)

## PINN - Poisson


In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.quasirandom import SobolEngine
import matplotlib.colors as mcolors
torch.manual_seed(88)
np.random.seed(88)

if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")

# ============================================================
# 2) Amostrador de dados esparsos
# ============================================================

def sample_sparse_data_from_reference(x, y, psi, omega, u, v,
                                       n_data=1000, seed=42):
    rng = np.random.default_rng(seed)
    Xg, Yg = np.meshgrid(x, y, indexing="xy")

    # Exclui os pontos de contorno da amostragem
    mask = np.ones_like(Xg, dtype=bool)
    mask[0, :] = mask[-1, :] = mask[:, 0] = mask[:, -1] = False

    data = np.column_stack([
        Xg[mask], Yg[mask],
        psi[mask], omega[mask], u[mask], v[mask],
    ])
    idx  = rng.choice(data.shape[0], size=n_data, replace=False)
    data = data[idx]

    return {
        "xy":    data[:, 0:2],
        "psi":   data[:, 2:3],
        "omega": data[:, 3:4],
        "u":     data[:, 4:5],
        "v":     data[:, 5:6],
    }


# ============================================================
# CORREÇÃO 1 – Embedding de Fourier aleatório (Random Fourier Features)
# ============================================================

class FourierEmbedding(nn.Module):
    """
    Mapeia (x, y) → [sin(Bx), cos(Bx)], com B ~ N(0, σ²).
    Permite que a rede represente variações espaciais de alta frequência
    (por exemplo, a fina camada de vorticidade sob a tampa móvel).
    """
    def __init__(self, in_dim: int = 2, n_freq: int = 32, sigma: float = 2.0):
        super().__init__()
        B = torch.randn(in_dim, n_freq) * sigma
        self.register_buffer("B", B)          # buffer fixo, não é parâmetro treinável

    @property
    def out_dim(self):
        return 2 * self.B.shape[1]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        proj = x @ self.B                     # (N, n_freq)
        return torch.cat([torch.sin(proj),
                          torch.cos(proj)], dim=-1)   # (N, 2*n_freq)


def build_mlp(layer_sizes):
    modules = []
    for i in range(len(layer_sizes) - 2):
        modules += [nn.Linear(layer_sizes[i], layer_sizes[i + 1]), nn.Tanh()]
    modules.append(nn.Linear(layer_sizes[-2], layer_sizes[-1]))
    return nn.Sequential(*modules)


# ============================================================
# CORREÇÃO 2 – Rede mais profunda + entrada de Fourier + normalização da entrada
# ============================================================

class CavityPINN(nn.Module):
    """
    Retorna (ψ, ω). As velocidades são obtidas por u = ∂ψ/∂y, v = -∂ψ/∂x.

    Melhorias
    ---------
    * As coordenadas de entrada são normalizadas para [-1, 1] antes de entrar na rede
    * Um embedding de Fourier substitui a entrada 2-D bruta
    * MLP mais larga/profunda: 4 camadas ocultas de 128 neurônios
    """
    def __init__(
        self,
        hidden_layers = (128, 128, 128, 128),
        n_freq: int   = 32,
        sigma: float  = 2.0,
        Lx: float     = 1.0,
        Ly: float     = 1.0,
    ):
        super().__init__()
        # CORREÇÃO 1: buffers de normalização da entrada
        self.register_buffer("x_mean", torch.tensor([Lx / 2.0, Ly / 2.0]))
        self.register_buffer("x_std",  torch.tensor([Lx / 2.0, Ly / 2.0]))

        # CORREÇÃO 2: embedding de Fourier
        self.fourier = FourierEmbedding(2, n_freq, sigma)
        in_dim = self.fourier.out_dim

        layer_sizes = [in_dim] + list(hidden_layers) + [2]
        self.net = build_mlp(layer_sizes)

    def forward(self, xy: torch.Tensor):
        xy_norm = (xy - self.x_mean) / self.x_std    # [-1, 1]
        feats   = self.fourier(xy_norm)
        out     = self.net(feats)
        return out[:, 0:1], out[:, 1:2]              # ψ, ω


# ============================================================
# CORREÇÃO 3 – Cálculo eficiente de gradientes
# ============================================================

def predict_fields(model, xy):
    """Calcula (ψ, ω, u, v) e as derivadas parciais de primeira ordem de forma eficiente."""
    psi, omega = model(xy)

    # uma única passagem reversa para todas as parciais de primeira ordem de ψ
    grads_psi = torch.autograd.grad(
        psi, xy,
        grad_outputs=torch.ones_like(psi),
        create_graph=True,
    )[0]
    psi_x, psi_y = grads_psi[:, 0:1], grads_psi[:, 1:2]

    u =  psi_y
    v = -psi_x

    return psi, omega, u, v, psi_x, psi_y


def physics_residuals(model, xy, nu):
    """
    Poisson:      ψ_xx + ψ_yy + ω = 0
    Vorticidade:  u·ω_x + v·ω_y - ν·(ω_xx + ω_yy) = 0
    """
    psi, omega, u, v, psi_x, psi_y = predict_fields(model, xy)

    # Parciais de primeira ordem de ω em uma única passagem
    grads_omega = torch.autograd.grad(
        omega, xy,
        grad_outputs=torch.ones_like(omega),
        create_graph=True,
    )[0]
    omega_x, omega_y = grads_omega[:, 0:1], grads_omega[:, 1:2]

    # Parciais de segunda ordem
    psi_xx   = torch.autograd.grad(psi_x,   xy, torch.ones_like(psi_x),   create_graph=True)[0][:, 0:1]
    psi_yy   = torch.autograd.grad(psi_y,   xy, torch.ones_like(psi_y),   create_graph=True)[0][:, 1:2]
    omega_xx = torch.autograd.grad(omega_x, xy, torch.ones_like(omega_x), create_graph=True)[0][:, 0:1]
    omega_yy = torch.autograd.grad(omega_y, xy, torch.ones_like(omega_y), create_graph=True)[0][:, 1:2]

    r_poisson   = psi_xx + psi_yy + omega
    r_vorticity = u * omega_x + v * omega_y - nu * (omega_xx + omega_yy)

    return r_poisson, r_vorticity


# ============================================================
# 3) Geradores de pontos de colocação e de contorno
# ============================================================

def make_sobol_points(n_points, Lx, Ly, device, seed=123):
    sobol = SobolEngine(dimension=2, scramble=True, seed=seed)
    pts   = sobol.draw(n_points).to(device)
    pts[:, 0] *= Lx
    pts[:, 1] *= Ly
    return pts


def make_boundary_points(n_per_side, Lx, Ly, U_lid, device, seed=123):
    rng = np.random.default_rng(seed)

    xb = rng.random((n_per_side, 1)) * Lx;  yb = np.zeros((n_per_side, 1))
    xt = rng.random((n_per_side, 1)) * Lx;  yt = np.full((n_per_side, 1), Ly)
    xl = np.zeros((n_per_side, 1));          yl = rng.random((n_per_side, 1)) * Ly
    xr = np.full((n_per_side, 1), Lx);      yr = rng.random((n_per_side, 1)) * Ly

    xy_bc = np.vstack([
        np.hstack([xb, yb]), np.hstack([xt, yt]),
        np.hstack([xl, yl]), np.hstack([xr, yr]),
    ])
    u_bc = np.vstack([
        np.zeros((n_per_side, 1)),
        np.full((n_per_side, 1), U_lid),
        np.zeros((n_per_side, 1)),
        np.zeros((n_per_side, 1)),
    ])

    return (
        torch.tensor(xy_bc, dtype=torch.float32, device=device),
        torch.tensor(u_bc,  dtype=torch.float32, device=device),
    )


def to_device_tensor(arr, device):
    return torch.tensor(arr, dtype=torch.float32, device=device)


# ============================================================
# 4) Termos individuais da função de perda
# ============================================================

def loss_data(model, xy_d, psi_d, omega_d, u_d, v_d):
    xy_d = xy_d.detach().requires_grad_(True)
    psi_p, omega_p, u_p, v_p, _, _ = predict_fields(model, xy_d)
    return (
          nn.functional.mse_loss(u_p,     u_d)
        + nn.functional.mse_loss(v_p,     v_d)
        + 0.05 * nn.functional.mse_loss(psi_p,   psi_d)
        + 0.05 * nn.functional.mse_loss(omega_p, omega_d)
    )


def loss_physics(model, xy_f, nu):
    xy_f = xy_f.detach().requires_grad_(True)
    r1, r2 = physics_residuals(model, xy_f, nu)
    return torch.mean(r1 ** 2) + torch.mean(r2 ** 2)


def loss_boundary(model, xy_b, u_bc):
    """
    Imposição explícita de não-deslizamento em cada parede:
      - Inferior (y=0) : u=0, v=0, psi=0
      - Esquerda (x=0) : u=0, v=0, psi=0
      - Direita  (x=1) : u=0, v=0, psi=0
      - Superior (y=1) : u=U_lid, v=0, psi=0  (tampa móvel)
    """
    n = xy_b.shape[0] // 4   # pontos por parede (conforme construído em make_boundary_points)

    # Separa as quatro paredes na mesma ordem em que make_boundary_points as empilha:
    # inferior, superior, esquerda, direita
    xy_bottom = xy_b[0*n : 1*n].detach().requires_grad_(True)
    xy_top    = xy_b[1*n : 2*n].detach().requires_grad_(True)
    xy_left   = xy_b[2*n : 3*n].detach().requires_grad_(True)
    xy_right  = xy_b[3*n : 4*n].detach().requires_grad_(True)

    u_lid_target = u_bc[1*n : 2*n]   # somente o topo tem U_lid != 0

    # Predição dos campos em cada parede
    psi_bot, _, u_bot, v_bot, _, _ = predict_fields(model, xy_bottom)
    psi_top, _, u_top, v_top, _, _ = predict_fields(model, xy_top)
    psi_lft, _, u_lft, v_lft, _, _ = predict_fields(model, xy_left)
    psi_rgt, _, u_rgt, v_rgt, _, _ = predict_fields(model, xy_right)

    # Paredes com não-deslizamento: u=0, v=0, psi=0
    loss_no_slip = (
          torch.mean(u_bot**2) + torch.mean(v_bot**2) + torch.mean(psi_bot**2)
        + torch.mean(u_lft**2) + torch.mean(v_lft**2) + torch.mean(psi_lft**2)
        + torch.mean(u_rgt**2) + torch.mean(v_rgt**2) + torch.mean(psi_rgt**2)
    )

    # Tampa móvel: u=U_lid, v=0, psi=0
    loss_lid = (
          torch.mean((u_top - u_lid_target)**2)
        + torch.mean(v_top**2)
        + torch.mean(psi_top**2)
    )

    return loss_no_slip + loss_lid

# ============================================================
# CORREÇÕES 4 e 5 – Peso da física crescente + treino em duas fases (Adam → L-BFGS)
# ============================================================

def train_pinn(
    model,
    xy_data_all, psi_data_all, omega_data_all, u_data_all, v_data_all,
    xy_f_all, xy_b_all, u_bc_all,
    nu,
    # Fase Adam
    adam_epochs  = 3000,
    batch_data   = 200,
    batch_f      = 2048,
    batch_b      = 512,
    lr           = 1e-3,
    w_data       = 5.0,
    w_phys_start = 1.0,
    w_phys_end   = 10.0,   # CORREÇÃO 4: peso da física crescente
    w_bc         = 10.0,
    # Fase L-BFGS
    lbfgs_steps  = 60,     # CORREÇÃO 5: segunda fase de otimização
    lbfgs_max_iter = 200,
):
    device = xy_f_all.device
    Nd, Nf, Nb = xy_data_all.shape[0], xy_f_all.shape[0], xy_b_all.shape[0]

    # ---- Fase Adam ----
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.MultiStepLR(
        optimizer, milestones=[1000, 2000], gamma=0.3
    )

    history = {"total": [], "data": [], "physics": [], "bc": []}

    for epoch in range(adam_epochs):
        idx_d = torch.randint(0, Nd, (min(batch_data, Nd),), device=device)
        idx_f = torch.randint(0, Nf, (min(batch_f,  Nf),),  device=device)
        idx_b = torch.randint(0, Nb, (min(batch_b,  Nb),),  device=device)

        # CORREÇÃO 4: aumenta linearmente o peso da física
        t = epoch / max(adam_epochs - 1, 1)
        w_phys = w_phys_start + t * (w_phys_end - w_phys_start)

        optimizer.zero_grad()

        ld = loss_data(
            model,
            xy_data_all[idx_d], psi_data_all[idx_d], omega_data_all[idx_d],
            u_data_all[idx_d],  v_data_all[idx_d],
        )
        lp = loss_physics(model, xy_f_all[idx_f], nu)
        idx_b = torch.randint(0, Nb, (min(batch_b, Nb),), device=device)
        lb = loss_boundary(model, xy_b_all[idx_b], u_bc_all[idx_b])

        # Depois — amostra por parede para preservar a ordenação:
        n_wall = Nb // 4
        b_per_wall = min(batch_b // 4, n_wall)
        idx_b = torch.cat([
            torch.randint(i*n_wall, (i+1)*n_wall, (b_per_wall,), device=device)
            for i in range(4)
        ])
        lb = loss_boundary(model, xy_b_all[idx_b], u_bc_all[idx_b])

        loss = w_data * ld + w_phys * lp + w_bc * lb
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        history["total"].append(loss.item())
        history["data"].append(ld.item())
        history["physics"].append(lp.item())
        history["bc"].append(lb.item())

        if epoch % 300 == 0:
            print(
                f"  Adam {epoch:5d}/{adam_epochs} | "
                f"Total={loss.item():.3e} | "
                f"Dados={ld.item():.3e} | "
                f"Física={lp.item():.3e} (w={w_phys:.1f}) | "
                f"CC={lb.item():.3e}"
            )

    # ---- CORREÇÃO 5: fase L-BFGS (batch completo) ----
    print("\nIniciando fase L-BFGS...")
    optimizer_lbfgs = torch.optim.LBFGS(
        model.parameters(),
        lr=0.1,
        max_iter=lbfgs_max_iter,
        tolerance_grad=1e-9,
        tolerance_change=1e-12,
        history_size=100,
        line_search_fn="strong_wolfe",
    )

    lbfgs_history = {"total": [], "data": [], "physics": [], "bc": []}

    def closure():
        optimizer_lbfgs.zero_grad()
        ld = loss_data(
            model,
            xy_data_all, psi_data_all, omega_data_all,
            u_data_all, v_data_all,
        )
        lp = loss_physics(model, xy_f_all, nu)
        lb = loss_boundary(model, xy_b_all, u_bc_all)
        total = w_data * ld + w_phys_end * lp + w_bc * lb
        total.backward()
        lbfgs_history["total"].append(total.item())
        lbfgs_history["data"].append(ld.item())
        lbfgs_history["physics"].append(lp.item())
        lbfgs_history["bc"].append(lb.item())
        return total

    for step in range(lbfgs_steps):
        optimizer_lbfgs.step(closure)
        if len(lbfgs_history["total"]) > 0:
            print(
                f"  L-BFGS passo {step + 1:3d}/{lbfgs_steps} | "
                f"Total={lbfgs_history['total'][-1]:.3e} | "
                f"Dados={lbfgs_history['data'][-1]:.3e} | "
                f"Física={lbfgs_history['physics'][-1]:.3e} | "
                f"CC={lbfgs_history['bc'][-1]:.3e}"
            )

    # junta os históricos
    for k in history:
        history[k] += lbfgs_history[k]

    return history


# ============================================================
# 5) Avaliação na malha
# ============================================================

def evaluate_pinn_on_grid(model, x, y, device):
    Xg, Yg = np.meshgrid(x, y, indexing="xy")
    xy = np.column_stack([Xg.ravel(), Yg.ravel()])
    xy_t = torch.tensor(xy, dtype=torch.float32, device=device).requires_grad_(True)

    with torch.enable_grad():
        psi_p, omega_p, u_p, v_p, _, _ = predict_fields(model, xy_t)

    def to_grid(t):
        return t.detach().cpu().numpy().reshape(Xg.shape)

    return Xg, Yg, to_grid(psi_p), to_grid(omega_p), to_grid(u_p), to_grid(v_p)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {device}\n")

# ----------------------------------------------------------
# Dados esparsos da CFD
# ----------------------------------------------------------
sparse = sample_sparse_data_from_reference(
    x=x, y=y,
    psi=psi, omega=omega, u=u, v=v,
    n_data=3600, seed=8,
)#123

xy_data_all    = to_device_tensor(sparse["xy"],    device)
psi_data_all   = to_device_tensor(sparse["psi"],   device)
omega_data_all = to_device_tensor(sparse["omega"], device)
u_data_all     = to_device_tensor(sparse["u"],     device)
v_data_all     = to_device_tensor(sparse["v"],     device)

# ----------------------------------------------------------
# Pontos de colocação e de contorno
Lx, Ly  = 1.0, 1.0
Re = 1000
U_lid = -1
nu = abs(U_lid) * Lx / Re
# ----------------------------------------------------------
xy_f_all         = make_sobol_points(2048, Lx, Ly, device, seed=188)#321
xy_b_all, u_bc_all = make_boundary_points(256, Lx, Ly, U_lid, device, seed=88)#456

# ----------------------------------------------------------
# Modelo  (CORREÇÃO 1 + CORREÇÃO 2)
# ----------------------------------------------------------


model = CavityPINN(
    hidden_layers=(128, 128, 128, 128),
    n_freq=48,
    sigma=4.0,
    Lx=Lx, Ly=Ly,
).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parâmetros da PINN: {total_params:,}\n")

# ----------------------------------------------------------
# Treinamento  (CORREÇÃO 4 + CORREÇÃO 5)
# ----------------------------------------------------------
print("Treinando a PINN...")
history = train_pinn(
    model=model,
    xy_data_all=xy_data_all,
    psi_data_all=psi_data_all,
    omega_data_all=omega_data_all,
    u_data_all=u_data_all,
    v_data_all=v_data_all,
    xy_f_all=xy_f_all,
    xy_b_all=xy_b_all,
    u_bc_all=u_bc_all,
    nu=nu,
    adam_epochs=5100,
    batch_data=200,
    batch_f=512,
    batch_b=512,
    lr=1e-3,
    w_data=40.0,#10
    w_phys_start=1.0,
    w_phys_end=40.0,#10
    w_bc=40,#8
    lbfgs_steps=40,
    lbfgs_max_iter=200,
)
print("Treinamento concluído.\n")

In [ ]:
# ----------------------------------------------------------
# Evaluate on full grid
# ----------------------------------------------------------
model.eval()
Xp, Yp, psi_pinn, omega_pinn, u_pinn, v_pinn = \
    evaluate_pinn_on_grid(model, x, y, device)

ix_center = len(x) // 2
iy_center = len(y) // 2

u_ref = u
v_ref = v
omega_ref = omega
psi_ref = psi
# ---- Plot 1: Streamlines side-by-side ----
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, u_plot, v_plot, title in zip(
    axes,
    [u_ref,  u_pinn],
    [v_ref,  v_pinn],
    ["Reference CFD", "Improved PINN"],
):
    ax.streamplot(X, Y, u_plot, v_plot, density=2.0, linewidth=0.9, arrowsize=0.9)
    ax.set_title(title)
    ax.set_xlabel("x"); ax.set_ylabel("y")
    ax.set_xlim(0, 1);  ax.set_ylim(0, 1)
    ax.set_aspect("equal")
plt.tight_layout()
plt.show()

# ---- Plot 2: u along central vertical line (x=0.5) ----
plt.figure(figsize=(6, 5))
plt.plot(u_ref[:, ix_center],  y, "k-",  linewidth=2.0, label="Reference CFD")
plt.plot(u_pinn[:, ix_center], y, "r--", linewidth=2.0, label="Improved PINN")
plt.title("u-velocity — central vertical line (x = 0.5)")
plt.xlabel("u"); plt.ylabel("y")
plt.grid(True, alpha=0.3); plt.legend()
plt.tight_layout()
plt.show()

# ---- Plot 3: u along central horizontal line (y=0.5) ----
plt.figure(figsize=(6, 5))
plt.plot(x, u_ref[iy_center, :],  "k-",  linewidth=2.0, label="Reference CFD")
plt.plot(x, u_pinn[iy_center, :], "r--", linewidth=2.0, label="Improved PINN")
plt.title("u-velocity — central horizontal line (y = 0.5)")
plt.xlabel("x"); plt.ylabel("u")
plt.grid(True, alpha=0.3); plt.legend()
plt.tight_layout()
plt.show()

# ---- Plot 4: v along central vertical line (x=0.5) ----
plt.figure(figsize=(6, 5))
plt.plot(v_ref[:, ix_center],  y, "k-",  linewidth=2.0, label="Reference CFD")
plt.plot(v_pinn[:, ix_center], y, "r--", linewidth=2.0, label="Improved PINN")
plt.title("v-velocity — central vertical line (x = 0.5)")
plt.xlabel("v"); plt.ylabel("y")
plt.grid(True, alpha=0.3); plt.legend()
plt.tight_layout()
plt.show()

# ---- Plot 5: Loss history ----
plt.figure(figsize=(8, 5))
plt.semilogy(history["total"],   label="Total",   linewidth=1.5)
plt.semilogy(history["data"],    label="Data",    linewidth=1.5)
plt.semilogy(history["physics"], label="Physics", linewidth=1.5)
plt.semilogy(history["bc"],      label="BC",      linewidth=1.5)
plt.axvline(3000, color="gray", linestyle=":", linewidth=1.0, label="Adam → L-BFGS")
plt.title("Training loss history")
plt.xlabel("Iteration"); plt.ylabel("Loss (log scale)")
plt.grid(True, alpha=0.3); plt.legend()
plt.tight_layout()
plt.show()

# ---- L2 error report ----
def rel_l2(pred, ref):
    return np.linalg.norm(pred - ref) / (np.linalg.norm(ref) + 1e-12)

print("\n--- Relative L2 errors on full grid ---")
print(f"  u    : {rel_l2(u_pinn,     u_ref)     * 100:.2f} %")
print(f"  v    : {rel_l2(v_pinn,     v_ref)     * 100:.2f} %")
print(f"  ψ    : {rel_l2(psi_pinn,   psi_ref)   * 100:.2f} %")
print(f"  ω    : {rel_l2(omega_pinn, omega_ref) * 100:.2f} %")

# ============================================================
# 7) Statistical verification of U_lid and Re from PINN fields
# ============================================================


print("\n--- Statistical verification of U_lid and Re from PINN fields ---")

u_top_pinn = u_pinn[-1, :]  
u_top_ref  = u_ref[-1, :]

U_lid_pinn = float(np.mean(u_top_pinn))
U_lid_ref  = float(np.mean(u_top_ref))
U_lid_std  = float(np.std(u_top_pinn))

print(f"\n  U_lid (original)       : {U_lid:.4f}")
print(f"  U_lid from CFD ref     : {U_lid_ref:.4f}")
print(f"  U_lid from PINN fields : {U_lid_pinn:.4f}  (std = {U_lid_std:.4f})")
print(f"  Relative error vs CFD  : {abs(U_lid_pinn - U_lid_ref) / abs(U_lid_ref) * 100:.2f} %")

dx_g = x[1] - x[0]
dy_g = y[1] - y[0]

u_i     = u_pinn[1:-1, 1:-1]
v_i     = v_pinn[1:-1, 1:-1]
omega_i = omega_pinn[1:-1, 1:-1]


domega_dx = (omega_pinn[1:-1, 2:] - omega_pinn[1:-1, :-2]) / (2 * dx_g)
domega_dy = (omega_pinn[2:, 1:-1] - omega_pinn[:-2, 1:-1]) / (2 * dy_g)

lap_omega = (
    (omega_pinn[1:-1, 2:] - 2*omega_pinn[1:-1, 1:-1] + omega_pinn[1:-1, :-2]) / dx_g**2
    + (omega_pinn[2:, 1:-1] - 2*omega_pinn[1:-1, 1:-1] + omega_pinn[:-2, 1:-1]) / dy_g**2
)


advection = (u_i * domega_dx + v_i * domega_dy).ravel()
diffusion  = lap_omega.ravel()


mask_valid = np.abs(diffusion) > 1e-6 
nu_fit = np.dot(diffusion[mask_valid], advection[mask_valid]) / \
        np.dot(diffusion[mask_valid], diffusion[mask_valid])

Re_pinn = U_lid_pinn * Lx / nu_fit
Re_ref  = U_lid * Lx / nu  

print(f"\n  Re (original)          : {Re:.2f}")
print(f"  nu fitted from PINN    : {nu_fit:.5f}  (original nu = {nu:.5f})")
print(f"  Re from PINN fields    : {Re_pinn:.2f}")
print(f"  Relative error vs CFD  : {abs(Re_pinn - Re_ref) / Re_ref * 100:.2f} %")

# ---- u profile at top wall ----
plt.figure(figsize=(7, 4))
plt.plot(x, u_top_ref,  "k-",  linewidth=2.0, label=f"CFD ref  (mean={U_lid_ref:.4f})")
plt.plot(x, u_top_pinn, "r--", linewidth=2.0, label=f"PINN     (mean={U_lid_pinn:.4f})")
plt.axhline(U_lid, color="gray", linestyle=":", linewidth=1.5, label=f"Target U_lid={U_lid}")
plt.title("u-velocity at top wall (y = 1.0) — U_lid verification")
plt.xlabel("x"); plt.ylabel("u")
plt.ylim(0.8, 1.2)
plt.grid(True, alpha=0.3); plt.legend()
plt.tight_layout()
plt.show()

# ---- Velocity modulus only (no streamlines) ----


speed_ref  = np.sqrt(u_ref**2  + v_ref**2)
speed_pinn = np.sqrt(u_pinn**2 + v_pinn**2)

vmin = 0.0
vmax = max(speed_ref.max(), speed_pinn.max())
norm = mcolors.Normalize(vmin=vmin, vmax=vmax)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, speed, title in zip(
    axes,
    [speed_ref, speed_pinn],
    ["Reference CFD", "Improved PINN"],
):
    im = ax.pcolormesh(
        X, Y, speed,
        cmap="plasma",
        norm=norm,
        shading="gouraud",  
    )
    ax.set_title(title)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect("equal")

fig.colorbar(im, ax=axes[1], label="|u| velocity modulus", fraction = 0.05)
plt.suptitle("Velocity modulus", y=1.02)
plt.tight_layout()
plt.show()

## Segunda Parte - Reynolds treinável e dados das Tabelas


In [ ]:
import torch
import torch.nn as nn
from torch.quasirandom import SobolEngine
import numpy as np
import matplotlib.colors as mcolors

torch.manual_seed(88)
np.random.seed(88)

if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")

# ============================================================
# 2) Sparse data sampler
# ============================================================

def sample_sparse_data_from_reference(x, y, u, v, p=None,
                                       n_data=1000, seed=42):
    """
    Sample sparse data for u, v (and optionally p) from the reference CFD.
    Pressure p is often optional in PINNs (Hidden Fluid Mechanics).
    """
    rng = np.random.default_rng(seed)
    Xg, Yg = np.meshgrid(x, y, indexing="xy")

    mask = np.ones_like(Xg, dtype=bool)
    mask[0, :] = mask[-1, :] = mask[:, 0] = mask[:, -1] = False

    if p is not None:
        data = np.column_stack([Xg[mask], Yg[mask], u[mask], v[mask], p[mask]])
    else:
        data = np.column_stack([Xg[mask], Yg[mask], u[mask], v[mask]])
        
    idx  = rng.choice(data.shape[0], size=n_data, replace=False)
    data = data[idx]

    batch = {
        "xy": data[:, 0:2],
        "u":  data[:, 2:3],
        "v":  data[:, 3:4],
    }
    if p is not None:
        batch["p"] = data[:, 4:5]
        
    return batch


# ============================================================
# FIX 1 – Random Fourier Feature embedding
# ============================================================

class FourierEmbedding(nn.Module):
    """
    Maps (x, y) → [sin(Bx), cos(Bx)]  where B ~ N(0, σ²).
    Lets the network represent high-frequency spatial variations.
    """
    def __init__(self, in_dim: int = 2, n_freq: int = 32, sigma: float = 2.0):
        super().__init__()
        B = torch.randn(in_dim, n_freq) * sigma
        self.register_buffer("B", B)          # not a learnable param

    @property
    def out_dim(self):
        return 2 * self.B.shape[1]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        proj = x @ self.B                     # (N, n_freq)
        return torch.cat([torch.sin(proj),
                          torch.cos(proj)], dim=-1)   # (N, 2*n_freq)


def build_mlp(layer_sizes):
    modules = []
    for i in range(len(layer_sizes) - 2):
        modules += [nn.Linear(layer_sizes[i], layer_sizes[i + 1]), nn.Tanh()]
    modules.append(nn.Linear(layer_sizes[-2], layer_sizes[-1]))
    return nn.Sequential(*modules)


# ============================================================
# FIX 2 – Deeper network + Fourier input + input normalisation
# ============================================================

class CavityPINN_NS(nn.Module):
    """
    Outputs (u, v, p).
    """
    def __init__(
        self,
        hidden_layers = (128, 128, 128, 128),
        n_freq: int   = 32,
        sigma: float  = 2.0,
        Lx: float     = 1.0,
        Ly: float     = 1.0,
        U_ref: float  = 1.0,
        Re_init: float = 500.0,
    ):
        super().__init__()
        # Input normalisation buffers
        self.register_buffer("x_mean", torch.tensor([Lx / 2.0, Ly / 2.0]))
        self.register_buffer("x_std",  torch.tensor([Lx / 2.0, Ly / 2.0]))

        # Known reference scales used in Re = U_ref * L_ref / nu
        self.register_buffer("U_ref", torch.tensor(float(abs(U_ref))))
        self.register_buffer("L_ref", torch.tensor(float(Lx)))

        # Inverse parameter. Optimizing log(Re) guarantees Re > 0.
        if Re_init <= 0:
            raise ValueError("Re_init must be positive.")
        self.log_Re = nn.Parameter(
            torch.tensor(np.log(Re_init), dtype=torch.float32)
        )

        # Fourier embedding
        self.fourier = FourierEmbedding(2, n_freq, sigma)
        in_dim = self.fourier.out_dim

        # Output is now 3: u, v, p
        layer_sizes = [in_dim] + list(hidden_layers) + [3]
        self.net = build_mlp(layer_sizes)

    def reynolds_number(self):
        """Current positive Reynolds-number estimate."""
        return torch.exp(self.log_Re)

    def kinematic_viscosity(self):
        """nu = U_ref * L_ref / Re."""
        return self.U_ref * self.L_ref / self.reynolds_number()

    def forward(self, xy: torch.Tensor):
        xy_norm = (xy - self.x_mean) / self.x_std   
        feats   = self.fourier(xy_norm)
        out     = self.net(feats)
        return out[:, 0:1], out[:, 1:2], out[:, 2:3] # u, v, p


# ============================================================
# FIX 3 – Efficient gradient computation (Navier-Stokes)
# ============================================================

def physics_residuals(model, xy):
    """
    Continuity: u_x + v_y = 0
    X-Momentum: u*u_x + v*u_y + p_x - ν*(u_xx + u_yy) = 0
    Y-Momentum: u*v_x + v*v_y + p_y - ν*(v_xx + v_yy) = 0
    """
    # 1. Forward pass
    u, v, p = model(xy)

    # 2. First-order derivatives (Compute all at once for each output to save backward passes)
    u_grad = torch.autograd.grad(u, xy, grad_outputs=torch.ones_like(u), create_graph=True)[0]
    u_x, u_y = u_grad[:, 0:1], u_grad[:, 1:2]

    v_grad = torch.autograd.grad(v, xy, grad_outputs=torch.ones_like(v), create_graph=True)[0]
    v_x, v_y = v_grad[:, 0:1], v_grad[:, 1:2]

    p_grad = torch.autograd.grad(p, xy, grad_outputs=torch.ones_like(p), create_graph=True)[0]
    p_x, p_y = p_grad[:, 0:1], p_grad[:, 1:2]

    # 3. Second-order derivatives (Only need diagonals: u_xx, u_yy, v_xx, v_yy)
    u_x_grad = torch.autograd.grad(u_x, xy, grad_outputs=torch.ones_like(u_x), create_graph=True)[0]
    u_xx = u_x_grad[:, 0:1]
    
    u_y_grad = torch.autograd.grad(u_y, xy, grad_outputs=torch.ones_like(u_y), create_graph=True)[0]
    u_yy = u_y_grad[:, 1:2]

    v_x_grad = torch.autograd.grad(v_x, xy, grad_outputs=torch.ones_like(v_x), create_graph=True)[0]
    v_xx = v_x_grad[:, 0:1]

    v_y_grad = torch.autograd.grad(v_y, xy, grad_outputs=torch.ones_like(v_y), create_graph=True)[0]
    v_yy = v_y_grad[:, 1:2]

    # 4. Navier-Stokes Residuals
    nu = model.kinematic_viscosity()
    r_cont  = u_x + v_y
    r_mom_x = u * u_x + v * u_y + p_x - nu * (u_xx + u_yy)
    r_mom_y = u * v_x + v * v_y + p_y - nu * (v_xx + v_yy)

    return r_cont, r_mom_x, r_mom_y


# ============================================================
# 3) Collocation & boundary point generators
# ============================================================

def make_sobol_points(n_points, Lx, Ly, device, seed=123):
    sobol = SobolEngine(dimension=2, scramble=True, seed=seed)
    pts   = sobol.draw(n_points).to(device)
    pts[:, 0] *= Lx
    pts[:, 1] *= Ly
    return pts


def make_boundary_points(n_per_side, Lx, Ly, U_lid, device, seed=123):
    rng = np.random.default_rng(seed)

    xb = rng.random((n_per_side, 1)) * Lx;  yb = np.zeros((n_per_side, 1))
    xt = rng.random((n_per_side, 1)) * Lx;  yt = np.full((n_per_side, 1), Ly)
    xl = np.zeros((n_per_side, 1));          yl = rng.random((n_per_side, 1)) * Ly
    xr = np.full((n_per_side, 1), Lx);      yr = rng.random((n_per_side, 1)) * Ly

    xy_bc = np.vstack([
        np.hstack([xb, yb]), np.hstack([xt, yt]),
        np.hstack([xl, yl]), np.hstack([xr, yr]),
    ])
    u_bc = np.vstack([
        np.zeros((n_per_side, 1)),
        np.full((n_per_side, 1), U_lid),
        np.zeros((n_per_side, 1)),
        np.zeros((n_per_side, 1)),
    ])

    return (
        torch.tensor(xy_bc, dtype=torch.float32, device=device),
        torch.tensor(u_bc,  dtype=torch.float32, device=device),
    )

def to_device_tensor(arr, device):
    return torch.tensor(arr, dtype=torch.float32, device=device)


# ============================================================
# 4) Individual loss terms
# ============================================================

def loss_data(model, xy_d, u_d, v_d, p_d=None):
    xy_d = xy_d.detach().requires_grad_(True)
    u_p, v_p, p_p = model(xy_d)
    
    loss = nn.functional.mse_loss(u_p, u_d) + nn.functional.mse_loss(v_p, v_d)
    if p_d is not None:
        loss += 0.05 * nn.functional.mse_loss(p_p, p_d)
    return loss
def loss_pressure_gauge(model, device, x_ref=0.5, y_ref=0.5):
    """
    Pins the pressure at the cavity center to zero,
    removing the arbitrary constant in the pressure field.
    """
    xy_ref = torch.tensor([[x_ref, y_ref]], dtype=torch.float32, device=device)
    _, _, p_ref = model(xy_ref)
    return p_ref ** 2
def pressure_correction_step(model, xy, lambda_mct=1e-2, n_iter=5, tol=1e-6):
    """
    Applies the Modified Compressibility iterative pressure correction.

    Algorithm:
      1. Forward pass -> get u, v, p (initial p from network)
      2. Compute dilation D = u_x + v_y
      3. If |D| <= tol or iter >= limit: exit
      4. p_correction = -lambda * D
      5. p_new = p_old + p_correction
      6. Recompute gradients with updated p and repeat

    Returns corrected (u, v, p, r_cont, r_mom_x, r_mom_y)
    """
    xy = xy.detach().requires_grad_(True)

    # Initial forward pass
    u, v, p = model(xy)

    for _ in range(n_iter):
        # --- Compute dilation D = u_x + v_y ---
        u_grad = torch.autograd.grad(
            u, xy, grad_outputs=torch.ones_like(u), create_graph=True
        )[0]
        u_x, u_y = u_grad[:, 0:1], u_grad[:, 1:2]

        v_grad = torch.autograd.grad(
            v, xy, grad_outputs=torch.ones_like(v), create_graph=True
        )[0]
        v_x, v_y = v_grad[:, 0:1], v_grad[:, 1:2]

        D = u_x + v_y  

        # --- Check convergence ---
        if D.abs().mean().item() <= tol:
            break

  
        p_correction = -lambda_mct * D
        p = p + p_correction.detach()   

    # --- Final residuals with corrected pressure ---
    # Recompute all derivatives cleanly for residual loss
    xy_r = xy.detach().requires_grad_(True)
    u_r, v_r, _ = model(xy_r)
    # Use corrected pressure (detached from correction loop, kept differentiable via xy_r)
    _, _, p_r = model(xy_r)
    p_r = p_r + (-lambda_mct * (
        torch.autograd.grad(u_r, xy_r, grad_outputs=torch.ones_like(u_r), create_graph=True)[0][:, 0:1] +
        torch.autograd.grad(v_r, xy_r, grad_outputs=torch.ones_like(v_r), create_graph=True)[0][:, 1:2]
    )).detach()

    p_grad = torch.autograd.grad(
        p_r, xy_r, grad_outputs=torch.ones_like(p_r), create_graph=True
    )[0]
    p_x, p_y = p_grad[:, 0:1], p_grad[:, 1:2]

    u_grad2 = torch.autograd.grad(u_r, xy_r, grad_outputs=torch.ones_like(u_r), create_graph=True)[0]
    u_x2, u_y2 = u_grad2[:, 0:1], u_grad2[:, 1:2]

    v_grad2 = torch.autograd.grad(v_r, xy_r, grad_outputs=torch.ones_like(v_r), create_graph=True)[0]
    v_x2, v_y2 = v_grad2[:, 0:1], v_grad2[:, 1:2]

    u_xx = torch.autograd.grad(u_x2, xy_r, grad_outputs=torch.ones_like(u_x2), create_graph=True)[0][:, 0:1]
    u_yy = torch.autograd.grad(u_y2, xy_r, grad_outputs=torch.ones_like(u_y2), create_graph=True)[0][:, 1:2]
    v_xx = torch.autograd.grad(v_x2, xy_r, grad_outputs=torch.ones_like(v_x2), create_graph=True)[0][:, 0:1]
    v_yy = torch.autograd.grad(v_y2, xy_r, grad_outputs=torch.ones_like(v_y2), create_graph=True)[0][:, 1:2]

    # Reynolds number is inferred by the PINN; viscosity follows from it.
    nu = model.kinematic_viscosity()

    # Modified continuity: lambda * dp/dt + D = 0 (enforced as D -> 0)
    r_cont  = u_x2 + v_y2              
    r_mom_x = u_r * u_x2 + v_r * u_y2 + p_x - nu * (u_xx + u_yy)
    r_mom_y = u_r * v_x2 + v_r * v_y2 + p_y - nu * (v_xx + v_yy)

    return r_cont, r_mom_x, r_mom_y, D.detach()


def loss_physics_mct(model, xy_f, lambda_mct=1e-2, n_iter=5, tol=1e-6):
    """
    Physics loss using the Modified Compressibility Technique.
    Adds a dilation penalty term to explicitly drive D -> 0.
    """
    r_cont, r_mom_x, r_mom_y, D = pressure_correction_step(
        model, xy_f,
        lambda_mct=lambda_mct,
        n_iter=n_iter,
        tol=tol,
    )

    loss_continuity = torch.mean(r_cont ** 2)
    loss_momentum   = torch.mean(r_mom_x ** 2) + torch.mean(r_mom_y ** 2)
    loss_dilation   = torch.mean(D ** 2)   # explicit penalty on divergence

    return loss_continuity + loss_momentum + loss_dilation


def loss_boundary(model, xy_b, u_bc):
    """
    Explicit no-slip enforcement per wall:
      - Bottom (y=0) : u=0, v=0
      - Left   (x=0) : u=0, v=0
      - Right  (x=1) : u=0, v=0
      - Top    (y=1) : u=U_lid, v=0  (driven lid)
    """
    n = xy_b.shape[0] // 4 

    xy_bottom = xy_b[0*n : 1*n].detach().requires_grad_(True)
    xy_top    = xy_b[1*n : 2*n].detach().requires_grad_(True)
    xy_left   = xy_b[2*n : 3*n].detach().requires_grad_(True)
    xy_right  = xy_b[3*n : 4*n].detach().requires_grad_(True)

    u_lid_target = u_bc[1*n : 2*n]   

    # Predict fields
    u_bot, v_bot, _ = model(xy_bottom)
    u_top, v_top, _ = model(xy_top)
    u_lft, v_lft, _ = model(xy_left)
    u_rgt, v_rgt, _ = model(xy_right)

    # No-slip walls: u=0, v=0
    loss_no_slip = (
          torch.mean(u_bot**2) + torch.mean(v_bot**2)
        + torch.mean(u_lft**2) + torch.mean(v_lft**2)
        + torch.mean(u_rgt**2) + torch.mean(v_rgt**2)
    )

    # Driven lid: u=U_lid, v=0
    loss_lid = (
          torch.mean((u_top - u_lid_target)**2)
        + torch.mean(v_top**2)
    )

    return loss_no_slip + loss_lid

# ============================================================
# FIX 4+5 – Ramped physics weight + two-phase Adam → L-BFGS
# ============================================================

def train_pinn(
    model,
    xy_data_all, u_data_all, v_data_all, p_data_all,
    xy_f_all, xy_b_all, u_bc_all,
    adam_epochs  = 3000,
    batch_data   = 200,
    batch_f      = 2048,
    batch_b      = 512,
    lr           = 1e-3,
    w_data       = 5.0,
    w_phys_start = 1.0,
    w_phys_end   = 10.0,
    w_bc         = 10.0,
    w_gauge        = 10.0,
    lbfgs_steps  = 60,
    lbfgs_max_iter = 200,
):
    device = xy_f_all.device
    Nd, Nf, Nb = xy_data_all.shape[0], xy_f_all.shape[0], xy_b_all.shape[0]

    # ---- Adam phase ----
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.MultiStepLR(
        optimizer, milestones=[1000, 2000], gamma=0.3
    )

    history = {"total": [], "data": [], "physics": [], "bc": [], "gauge": [], "Re": []}

    for epoch in range(adam_epochs):
        idx_d = torch.randint(0, Nd, (min(batch_data, Nd),), device=device)
        idx_f = torch.randint(0, Nf, (min(batch_f,  Nf),),  device=device)

        t = epoch / max(adam_epochs - 1, 1)
        w_phys = w_phys_start + t * (w_phys_end - w_phys_start)

        optimizer.zero_grad()

        # Handle optional pressure
        p_batch = p_data_all[idx_d] if p_data_all is not None else None
        
        ld = loss_data(
            model,
            xy_data_all[idx_d], u_data_all[idx_d], v_data_all[idx_d], p_batch
        )
        lp = loss_physics_mct(
            model, xy_f_all[idx_f],
            lambda_mct=1e-2,   # small compressibility constant
            n_iter=5,          # inner correction iterations
            tol=1e-6,
        )

  
        n_wall = Nb // 4
        b_per_wall = min(batch_b // 4, n_wall)
        idx_b = torch.cat([
            torch.randint(i*n_wall, (i+1)*n_wall, (b_per_wall,), device=device)
            for i in range(4)
        ])
        lb = loss_boundary(model, xy_b_all[idx_b], u_bc_all[idx_b])

        lg = loss_pressure_gauge(model, device)

        loss = w_data * ld + w_phys * lp + w_bc * lb+ w_gauge * lg
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        history["total"].append(loss.item())
        history["data"].append(ld.item())
        history["physics"].append(lp.item())
        history["bc"].append(lb.item())

        history["gauge"].append(lg.item())
        history["Re"].append(model.reynolds_number().detach().item())

        if epoch % 300 == 0:
            print(
                f"  Adam {epoch:5d}/{adam_epochs} | "
                f"Total={loss.item():.3e} | "
                f"Data={ld.item():.3e} | "
                f"Phys={lp.item():.3e} (w={w_phys:.1f}) | "
                f"BC={lb.item():.3e} | "
                f"Gauge={lg.item():.3e} | "
                f"Re={model.reynolds_number().detach().item():.3f}"
            )

    # ---- FIX 5: L-BFGS phase (full-batch) ----
    print("\nStarting L-BFGS phase...")
    optimizer_lbfgs = torch.optim.LBFGS(
        model.parameters(),
        lr=0.1,
        max_iter=lbfgs_max_iter,
        tolerance_grad=1e-9,
        tolerance_change=1e-12,
        history_size=100,
        line_search_fn="strong_wolfe",
    )

    lbfgs_history = {"total": [], "data": [], "physics": [], "bc": [], "gauge": [], "Re": []}

    def closure():
        optimizer_lbfgs.zero_grad()
        ld = loss_data(
            model,
            xy_data_all, u_data_all, v_data_all, p_data_all
        )
        lp = loss_physics_mct(
            model, xy_f_all,
            lambda_mct=1e-2,   # small compressibility constant
            n_iter=5,          # inner correction iterations
            tol=1e-6,
        )
        lb = loss_boundary(model, xy_b_all, u_bc_all)
        lg = loss_pressure_gauge(model, device)
        total = w_data * ld + w_phys_end * lp + w_bc * lb + w_gauge * lg
        total.backward()
        lbfgs_history["total"].append(total.item())
        lbfgs_history["data"].append(ld.item())
        lbfgs_history["physics"].append(lp.item())
        lbfgs_history["bc"].append(lb.item())
        lbfgs_history["gauge"].append(lg.item())
        lbfgs_history["Re"].append(model.reynolds_number().detach().item())
        return total

    for step in range(lbfgs_steps):
        optimizer_lbfgs.step(closure)
        if len(lbfgs_history["total"]) > 0:
            print(
                f"  L-BFGS step {step + 1:3d}/{lbfgs_steps} | "
                f"Total={lbfgs_history['total'][-1]:.3e} | "
                f"Data={lbfgs_history['data'][-1]:.3e} | "
                f"Phys={lbfgs_history['physics'][-1]:.3e} | "
                f"BC={lbfgs_history['bc'][-1]:.3e} | "
                f"Re={model.reynolds_number().detach().item():.3f}"
            )

    for k in history:
        history[k] += lbfgs_history[k]

    return history


# ============================================================
# 5) Evaluate on grid
# ============================================================

def evaluate_pinn_on_grid(model, x, y, device):
    Xg, Yg = np.meshgrid(x, y, indexing="xy")
    xy = np.column_stack([Xg.ravel(), Yg.ravel()])
    xy_t = torch.tensor(xy, dtype=torch.float32, device=device).requires_grad_(True)

    with torch.no_grad():
        u_p, v_p, p_p = model(xy_t)

    def to_grid(t):
        return t.detach().cpu().numpy().reshape(Xg.shape)

    return Xg, Yg, to_grid(u_p), to_grid(v_p), to_grid(p_p)

# ============================================================
# Execution Entry Point
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}\n")

try:
    _p_ref = p  # if p exists
except NameError:
    _p_ref = None

# ----------------------------------------------------------
# Sparse data from CFD
# ----------------------------------------------------------
sparse = sample_sparse_data_from_reference(
    x=x, y=y,
    u=u, v=v, p=_p_ref,
    n_data=3600, seed=8,
)

xy_data_all    = to_device_tensor(sparse["xy"], device)
u_data_all     = to_device_tensor(sparse["u"],  device)
v_data_all     = to_device_tensor(sparse["v"],  device)
p_data_all     = to_device_tensor(sparse["p"],  device) if "p" in sparse else None

# ----------------------------------------------------------
# Collocation & boundary points
Lx, Ly  = 1.0, 1.0
U_lid = -1.0

# Reynolds number is UNKNOWN.
# This value is only the starting guess for the inverse problem.
Re_initial_guess = 500.0
# ----------------------------------------------------------
xy_f_all         = make_sobol_points(2048, Lx, Ly, device, seed=188)
xy_b_all, u_bc_all = make_boundary_points(256, Lx, Ly, U_lid, device, seed=88)

# ----------------------------------------------------------
# Model
# ----------------------------------------------------------
model = CavityPINN_NS(
    hidden_layers=(128, 128, 128, 128),
    n_freq=48,
    sigma=4.0,
    Lx=Lx, Ly=Ly,
    U_ref=abs(U_lid),
    Re_init=Re_initial_guess,
).to(device)

total_params = sum(param.numel() for param in model.parameters() if param.requires_grad)
print(f"PINN (Navier-Stokes) parameters: {total_params:,}\n")

# ----------------------------------------------------------
# Train
# ----------------------------------------------------------
print("Training PINN...")
history = train_pinn(
    model=model,
    xy_data_all=xy_data_all,
    u_data_all=u_data_all,
    v_data_all=v_data_all,
    p_data_all=p_data_all,
    xy_f_all=xy_f_all,
    xy_b_all=xy_b_all,
    u_bc_all=u_bc_all,
    adam_epochs=4500,
    batch_data=200,
    batch_f=512,
    batch_b=512,
    lr=1e-3,
    w_data=40.0,
    w_phys_start=1.0,
    w_phys_end=40.0,
    w_bc=40.0,
    w_gauge= 10.0,
    lbfgs_steps=80,
    lbfgs_max_iter=200,
)
print("Training done.\n")

# ----------------------------------------------------------
# Identified inverse parameter
# ----------------------------------------------------------
Re_identified = model.reynolds_number().detach().cpu().item()
nu_identified = model.kinematic_viscosity().detach().cpu().item()

print(f"Identified Reynolds number: Re = {Re_identified:.6f}")
print(f"Corresponding kinematic viscosity: nu = {nu_identified:.6e}")


# Sem Pressão

In [ ]:
import torch
import torch.nn as nn
from torch.quasirandom import SobolEngine
import numpy as np

torch.manual_seed(88)
np.random.seed(88)

if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")

# ============================================================
# 2) Dados experimentais/numericos fornecidos nas duas tabelas
# ============================================================

def load_centerline_velocity_data(source="calculado"):
    """Carrega exclusivamente valores de velocidade das imagens.

    Primeira tabela: v(x, y=0.5), 17 pontos.
    Segunda tabela: u(x=0.5, y), 17 pontos.

    "calculado"  -> colunas 'v' e 'u' (3as colunas das imagens).
    "referencia" -> colunas 'v, Ref. [12]' e 'u, Ref. [12]'.

    As duas componentes sao observadas em conjuntos de pontos diferentes.
    Nao se preenche a componente desconhecida com zero.
    """
    # Tabela 1 - Perfil horizontal, velocidade vertical v em y=0.5
    x_v = np.array([
        0.0000, 0.0312, 0.0391, 0.0469, 0.0547, 0.0937,
        0.1406, 0.1953, 0.5000, 0.7656, 0.7734, 0.8437,
        0.9062, 0.9219, 0.9297, 0.9375, 1.0000,
    ], dtype=np.float32)

    v_calculado = np.array([
         0.0000000, -0.2279225, -0.2936869, -0.3553213,
        -0.4103754, -0.5264392, -0.4264545, -0.3202137,
         0.0257995,  0.3253592,  0.3339924,  0.3769189,
         0.3330442,  0.3099097,  0.2962703,  0.2807056,
         0.0000000,
    ], dtype=np.float32)

    v_referencia = np.array([
         0.00000, -0.21388, -0.27669, -0.33714, -0.39188,
        -0.51550, -0.42665, -0.31966,  0.02526,  0.32235,
         0.33075,  0.37095,  0.32627,  0.30353,  0.29012,
         0.27485,  0.00000,
    ], dtype=np.float32)

    # Tabela 2 - Perfil vertical, velocidade horizontal u em x=0.5
    y_u = np.array([
        1.0000, 0.9766, 0.9688, 0.9609, 0.9531, 0.8516,
        0.7344, 0.6172, 0.5000, 0.4531, 0.2813, 0.1719,
        0.1016, 0.0703, 0.0625, 0.0547, 0.0000,
    ], dtype=np.float32)

    u_calculado = np.array([
        -1.0000000, -0.6644227, -0.5808359, -0.5169277,
        -0.4723329, -0.3372212, -0.1886747, -0.0570178,
         0.0620561,  0.1081999,  0.2803696,  0.3885691,
         0.3004561,  0.2228955,  0.2023300,  0.1812881,
         0.0000000,
    ], dtype=np.float32)

    u_referencia = np.array([
        -1.00000, -0.65928, -0.57492, -0.51117, -0.46604,
        -0.33304, -0.18719, -0.05702,  0.06080,  0.10648,
         0.27805,  0.38289,  0.29730,  0.22220,  0.20196,
         0.18109,  0.00000,
    ], dtype=np.float32)

    if source == "calculado":
        u_obs, v_obs = u_calculado, v_calculado
    elif source == "referencia":
        u_obs, v_obs = u_referencia, v_referencia
    else:
        raise ValueError("source deve ser 'calculado' ou 'referencia'.")

    xy_u = np.column_stack((np.full_like(y_u, 0.5), y_u))
    xy_v = np.column_stack((x_v, np.full_like(x_v, 0.5)))

    assert len(xy_u) == len(u_obs) == 17
    assert len(xy_v) == len(v_obs) == 17

    return {"xy_u": xy_u, "u": u_obs[:, None],
            "xy_v": xy_v, "v": v_obs[:, None]}


# ============================================================
# FIX 1 – Random Fourier Feature embedding
# ============================================================

class FourierEmbedding(nn.Module):
    """
    Maps (x, y) → [sin(Bx), cos(Bx)]  where B ~ N(0, σ²).
    Lets the network represent high-frequency spatial variations.
    """
    def __init__(self, in_dim: int = 2, n_freq: int = 32, sigma: float = 2.0):
        super().__init__()
        B = torch.randn(in_dim, n_freq) * sigma
        self.register_buffer("B", B)          # not a learnable param

    @property
    def out_dim(self):
        return 2 * self.B.shape[1]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        proj = x @ self.B                     # (N, n_freq)
        return torch.cat([torch.sin(proj),
                          torch.cos(proj)], dim=-1)   # (N, 2*n_freq)


def build_mlp(layer_sizes):
    modules = []
    for i in range(len(layer_sizes) - 2):
        modules += [nn.Linear(layer_sizes[i], layer_sizes[i + 1]), nn.Tanh()]
    modules.append(nn.Linear(layer_sizes[-2], layer_sizes[-1]))
    return nn.Sequential(*modules)


# ============================================================
# FIX 2 – Deeper network + Fourier input + input normalisation
# ============================================================

class CavityPINN_NS(nn.Module):
    """
    Outputs (u, v, p).
    """
    def __init__(
        self,
        hidden_layers = (128, 128, 128, 128),
        n_freq: int   = 32,
        sigma: float  = 2.0,
        Lx: float     = 1.0,
        Ly: float     = 1.0,
        U_ref: float  = 1.0,
        Re_init: float = 500.0,
    ):
        super().__init__()
        # Input normalisation buffers
        self.register_buffer("x_mean", torch.tensor([Lx / 2.0, Ly / 2.0]))
        self.register_buffer("x_std",  torch.tensor([Lx / 2.0, Ly / 2.0]))

        # Known reference scales used in Re = U_ref * L_ref / nu
        self.register_buffer("U_ref", torch.tensor(float(abs(U_ref))))
        self.register_buffer("L_ref", torch.tensor(float(Lx)))

        # Inverse parameter. Optimizing log(Re) guarantees Re > 0.
        if Re_init <= 0:
            raise ValueError("Re_init must be positive.")
        self.log_Re = nn.Parameter(
            torch.tensor(np.log(Re_init), dtype=torch.float32)
        )

        # Fourier embedding
        self.fourier = FourierEmbedding(2, n_freq, sigma)
        in_dim = self.fourier.out_dim

        # Output is now 3: u, v, p
        layer_sizes = [in_dim] + list(hidden_layers) + [3]
        self.net = build_mlp(layer_sizes)

    def reynolds_number(self):
        """Current positive Reynolds-number estimate."""
        return torch.exp(self.log_Re)

    def kinematic_viscosity(self):
        """nu = U_ref * L_ref / Re."""
        return self.U_ref * self.L_ref / self.reynolds_number()

    def forward(self, xy: torch.Tensor):
        xy_norm = (xy - self.x_mean) / self.x_std    # [-1, 1]
        feats   = self.fourier(xy_norm)
        out     = self.net(feats)
        return out[:, 0:1], out[:, 1:2], out[:, 2:3] # u, v, p


# ============================================================
# FIX 3 – Efficient gradient computation (Navier-Stokes)
# ============================================================

def physics_residuals(model, xy):
    """
    Continuity: u_x + v_y = 0
    X-Momentum: u*u_x + v*u_y + p_x - ν*(u_xx + u_yy) = 0
    Y-Momentum: u*v_x + v*v_y + p_y - ν*(v_xx + v_yy) = 0
    """
    # 1. Forward pass
    u, v, p = model(xy)

    # 2. First-order derivatives (Compute all at once for each output to save backward passes)
    u_grad = torch.autograd.grad(u, xy, grad_outputs=torch.ones_like(u), create_graph=True)[0]
    u_x, u_y = u_grad[:, 0:1], u_grad[:, 1:2]

    v_grad = torch.autograd.grad(v, xy, grad_outputs=torch.ones_like(v), create_graph=True)[0]
    v_x, v_y = v_grad[:, 0:1], v_grad[:, 1:2]

    p_grad = torch.autograd.grad(p, xy, grad_outputs=torch.ones_like(p), create_graph=True)[0]
    p_x, p_y = p_grad[:, 0:1], p_grad[:, 1:2]

    # 3. Second-order derivatives (Only need diagonals: u_xx, u_yy, v_xx, v_yy)
    u_x_grad = torch.autograd.grad(u_x, xy, grad_outputs=torch.ones_like(u_x), create_graph=True)[0]
    u_xx = u_x_grad[:, 0:1]
    
    u_y_grad = torch.autograd.grad(u_y, xy, grad_outputs=torch.ones_like(u_y), create_graph=True)[0]
    u_yy = u_y_grad[:, 1:2]

    v_x_grad = torch.autograd.grad(v_x, xy, grad_outputs=torch.ones_like(v_x), create_graph=True)[0]
    v_xx = v_x_grad[:, 0:1]

    v_y_grad = torch.autograd.grad(v_y, xy, grad_outputs=torch.ones_like(v_y), create_graph=True)[0]
    v_yy = v_y_grad[:, 1:2]

    # 4. Navier-Stokes Residuals
    # Viscosity is obtained from the trainable Reynolds number.
    nu = model.kinematic_viscosity()
    r_cont  = u_x + v_y
    r_mom_x = u * u_x + v * u_y + p_x - nu * (u_xx + u_yy)
    r_mom_y = u * v_x + v * v_y + p_y - nu * (v_xx + v_yy)

    return r_cont, r_mom_x, r_mom_y


# ============================================================
# 3) Collocation & boundary point generators
# ============================================================

def make_sobol_points(n_points, Lx, Ly, device, seed=123):
    sobol = SobolEngine(dimension=2, scramble=True, seed=seed)
    pts   = sobol.draw(n_points).to(device)
    pts[:, 0] *= Lx
    pts[:, 1] *= Ly
    return pts


def make_boundary_points(n_per_side, Lx, Ly, U_lid, device, seed=123):
    rng = np.random.default_rng(seed)

    xb = rng.random((n_per_side, 1)) * Lx;  yb = np.zeros((n_per_side, 1))
    xt = rng.random((n_per_side, 1)) * Lx;  yt = np.full((n_per_side, 1), Ly)
    xl = np.zeros((n_per_side, 1));          yl = rng.random((n_per_side, 1)) * Ly
    xr = np.full((n_per_side, 1), Lx);      yr = rng.random((n_per_side, 1)) * Ly

    xy_bc = np.vstack([
        np.hstack([xb, yb]), np.hstack([xt, yt]),
        np.hstack([xl, yl]), np.hstack([xr, yr]),
    ])
    u_bc = np.vstack([
        np.zeros((n_per_side, 1)),
        np.full((n_per_side, 1), U_lid),
        np.zeros((n_per_side, 1)),
        np.zeros((n_per_side, 1)),
    ])

    return (
        torch.tensor(xy_bc, dtype=torch.float32, device=device),
        torch.tensor(u_bc,  dtype=torch.float32, device=device),
    )

def to_device_tensor(arr, device):
    return torch.tensor(arr, dtype=torch.float32, device=device)


# ============================================================
# 4) Individual loss terms
# ============================================================

def loss_data(model, xy_u, u_obs, xy_v, v_obs):
    """Perda supervisionada nos perfis centrais separados.

    Para cada ponto do perfil de u, apenas u e conhecido.
    Para cada ponto do perfil de v, apenas v e conhecido.
    """
    u_pred, _, _ = model(xy_u)
    _, v_pred, _ = model(xy_v)
    loss_u = nn.functional.mse_loss(u_pred, u_obs)
    loss_v = nn.functional.mse_loss(v_pred, v_obs)
    return loss_u + loss_v

# ============================================================
# Modified Compressibility Technique (MCT) for Pressure
# Based on: Dyn. eqs. with iterative pressure correction
#   p_correction = -lambda * D,  D = du/dx + dv/dy
#   p_new = p_old + p_correction
# ============================================================
def loss_pressure_gauge(model, device, x_ref=0.5, y_ref=0.5):
    """
    Pins the pressure at the cavity center to zero,
    removing the arbitrary constant in the pressure field.
    """
    xy_ref = torch.tensor([[x_ref, y_ref]], dtype=torch.float32, device=device)
    _, _, p_ref = model(xy_ref)
    return p_ref ** 2
def pressure_correction_step(model, xy, lambda_mct=1e-2, n_iter=5, tol=1e-6):
    """
    Applies the Modified Compressibility iterative pressure correction.

    Algorithm:
      1. Forward pass -> get u, v, p (initial p from network)
      2. Compute dilation D = u_x + v_y
      3. If |D| <= tol or iter >= limit: exit
      4. p_correction = -lambda * D
      5. p_new = p_old + p_correction
      6. Recompute gradients with updated p and repeat

    Returns corrected (u, v, p, r_cont, r_mom_x, r_mom_y)
    """
    xy = xy.detach().requires_grad_(True)

    # Initial forward pass
    u, v, p = model(xy)

    for _ in range(n_iter):
        # --- Compute dilation D = u_x + v_y ---
        u_grad = torch.autograd.grad(
            u, xy, grad_outputs=torch.ones_like(u), create_graph=True
        )[0]
        u_x, u_y = u_grad[:, 0:1], u_grad[:, 1:2]

        v_grad = torch.autograd.grad(
            v, xy, grad_outputs=torch.ones_like(v), create_graph=True
        )[0]
        v_x, v_y = v_grad[:, 0:1], v_grad[:, 1:2]

        D = u_x + v_y  # dilation / divergence of velocity

        # --- Check convergence ---
        if D.abs().mean().item() <= tol:
            break

        # --- Pressure correction: p_new = p_old - lambda * D ---
        p_correction = -lambda_mct * D
        p = p + p_correction.detach()   # detach correction to avoid graph issues

    # --- Final residuals with corrected pressure ---
    # Recompute all derivatives cleanly for residual loss
    xy_r = xy.detach().requires_grad_(True)
    u_r, v_r, _ = model(xy_r)
    # Use corrected pressure (detached from correction loop, kept differentiable via xy_r)
    _, _, p_r = model(xy_r)
    p_r = p_r + (-lambda_mct * (
        torch.autograd.grad(u_r, xy_r, grad_outputs=torch.ones_like(u_r), create_graph=True)[0][:, 0:1] +
        torch.autograd.grad(v_r, xy_r, grad_outputs=torch.ones_like(v_r), create_graph=True)[0][:, 1:2]
    )).detach()

    p_grad = torch.autograd.grad(
        p_r, xy_r, grad_outputs=torch.ones_like(p_r), create_graph=True
    )[0]
    p_x, p_y = p_grad[:, 0:1], p_grad[:, 1:2]

    u_grad2 = torch.autograd.grad(u_r, xy_r, grad_outputs=torch.ones_like(u_r), create_graph=True)[0]
    u_x2, u_y2 = u_grad2[:, 0:1], u_grad2[:, 1:2]

    v_grad2 = torch.autograd.grad(v_r, xy_r, grad_outputs=torch.ones_like(v_r), create_graph=True)[0]
    v_x2, v_y2 = v_grad2[:, 0:1], v_grad2[:, 1:2]

    u_xx = torch.autograd.grad(u_x2, xy_r, grad_outputs=torch.ones_like(u_x2), create_graph=True)[0][:, 0:1]
    u_yy = torch.autograd.grad(u_y2, xy_r, grad_outputs=torch.ones_like(u_y2), create_graph=True)[0][:, 1:2]
    v_xx = torch.autograd.grad(v_x2, xy_r, grad_outputs=torch.ones_like(v_x2), create_graph=True)[0][:, 0:1]
    v_yy = torch.autograd.grad(v_y2, xy_r, grad_outputs=torch.ones_like(v_y2), create_graph=True)[0][:, 1:2]

    # Reynolds number is inferred by the PINN; viscosity follows from it.
    nu = model.kinematic_viscosity()

    # Modified continuity: lambda * dp/dt + D = 0 (enforced as D -> 0)
    r_cont  = u_x2 + v_y2                                    # should -> 0 after correction
    r_mom_x = u_r * u_x2 + v_r * u_y2 + p_x - nu * (u_xx + u_yy)
    r_mom_y = u_r * v_x2 + v_r * v_y2 + p_y - nu * (v_xx + v_yy)

    return r_cont, r_mom_x, r_mom_y, D.detach()


def loss_physics_mct(model, xy_f, lambda_mct=1e-2, n_iter=5, tol=1e-6):
    """
    Physics loss using the Modified Compressibility Technique.
    Adds a dilation penalty term to explicitly drive D -> 0.
    """
    r_cont, r_mom_x, r_mom_y, D = pressure_correction_step(
        model, xy_f,
        lambda_mct=lambda_mct,
        n_iter=n_iter,
        tol=tol,
    )

    loss_continuity = torch.mean(r_cont ** 2)
    loss_momentum   = torch.mean(r_mom_x ** 2) + torch.mean(r_mom_y ** 2)
    loss_dilation   = torch.mean(D ** 2)   # explicit penalty on divergence

    return loss_continuity + loss_momentum + loss_dilation


def loss_boundary(model, xy_b, u_bc):
    """
    Explicit no-slip enforcement per wall:
      - Bottom (y=0) : u=0, v=0
      - Left   (x=0) : u=0, v=0
      - Right  (x=1) : u=0, v=0
      - Top    (y=1) : u=U_lid, v=0  (driven lid)
    """
    n = xy_b.shape[0] // 4   # points per side

    xy_bottom = xy_b[0*n : 1*n].detach().requires_grad_(True)
    xy_top    = xy_b[1*n : 2*n].detach().requires_grad_(True)
    xy_left   = xy_b[2*n : 3*n].detach().requires_grad_(True)
    xy_right  = xy_b[3*n : 4*n].detach().requires_grad_(True)

    u_lid_target = u_bc[1*n : 2*n]   # only the top has U_lid != 0

    # Predict fields
    u_bot, v_bot, _ = model(xy_bottom)
    u_top, v_top, _ = model(xy_top)
    u_lft, v_lft, _ = model(xy_left)
    u_rgt, v_rgt, _ = model(xy_right)

    # No-slip walls: u=0, v=0
    loss_no_slip = (
          torch.mean(u_bot**2) + torch.mean(v_bot**2)
        + torch.mean(u_lft**2) + torch.mean(v_lft**2)
        + torch.mean(u_rgt**2) + torch.mean(v_rgt**2)
    )

    # Driven lid: u=U_lid, v=0
    loss_lid = (
          torch.mean((u_top - u_lid_target)**2)
        + torch.mean(v_top**2)
    )

    return loss_no_slip + loss_lid

# ============================================================
# FIX 4+5 – Ramped physics weight + two-phase Adam → L-BFGS
# ============================================================

def train_pinn(
    model,
    xy_u_all, u_data_all, xy_v_all, v_data_all,
    xy_f_all, xy_b_all, u_bc_all,
    adam_epochs  = 3000,
    batch_f      = 2048,
    batch_b      = 512,
    lr           = 1e-3,
    w_data       = 5.0,
    w_phys_start = 1.0,
    w_phys_end   = 10.0,
    w_bc         = 10.0,
    w_gauge        = 10.0,
    lbfgs_steps  = 60,
    lbfgs_max_iter = 200,
):
    device = xy_f_all.device
    Nf, Nb = xy_f_all.shape[0], xy_b_all.shape[0]

    # ---- Adam phase ----
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.MultiStepLR(
        optimizer, milestones=[1000, 2000], gamma=0.3
    )

    history = {"total": [], "data": [], "physics": [], "bc": [], "gauge": [], "Re": []}

    for epoch in range(adam_epochs):
        idx_f = torch.randint(0, Nf, (min(batch_f,  Nf),),  device=device)

        t = epoch / max(adam_epochs - 1, 1)
        w_phys = w_phys_start + t * (w_phys_end - w_phys_start)

        optimizer.zero_grad()

        # Todos os 17 pontos de cada perfil entram em toda epoca.
        ld = loss_data(model, xy_u_all, u_data_all, xy_v_all, v_data_all)
        lp = loss_physics_mct(
            model, xy_f_all[idx_f],
            lambda_mct=1e-2,   # small compressibility constant
            n_iter=5,          # inner correction iterations
            tol=1e-6,
        )

        # Sample per wall to keep ordering intact:
        n_wall = Nb // 4
        b_per_wall = min(batch_b // 4, n_wall)
        idx_b = torch.cat([
            torch.randint(i*n_wall, (i+1)*n_wall, (b_per_wall,), device=device)
            for i in range(4)
        ])
        lb = loss_boundary(model, xy_b_all[idx_b], u_bc_all[idx_b])

        lg = loss_pressure_gauge(model, device)

        loss = w_data * ld + w_phys * lp + w_bc * lb + w_gauge * lg
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        history["total"].append(loss.item())
        history["data"].append(ld.item())
        history["physics"].append(lp.item())
        history["bc"].append(lb.item())

        history["gauge"].append(lg.item())
        history["Re"].append(model.reynolds_number().detach().item())

        if epoch % 300 == 0:
            print(
                f"  Adam {epoch:5d}/{adam_epochs} | "
                f"Total={loss.item():.3e} | "
                f"Data={ld.item():.3e} | "
                f"Phys={lp.item():.3e} (w={w_phys:.1f}) | "
                f"BC={lb.item():.3e} | "
                f"Gauge={lg.item():.3e} | "
                f"Re={model.reynolds_number().detach().item():.3f}"
            )

    # ---- FIX 5: L-BFGS phase (full-batch) ----
    print("\nStarting L-BFGS phase...")
    optimizer_lbfgs = torch.optim.LBFGS(
        model.parameters(),
        lr=0.1,
        max_iter=lbfgs_max_iter,
        tolerance_grad=1e-9,
        tolerance_change=1e-12,
        history_size=100,
        line_search_fn="strong_wolfe",
    )

    lbfgs_history = {"total": [], "data": [], "physics": [], "bc": [], "gauge": [], "Re": []}

    def closure():
        optimizer_lbfgs.zero_grad()
        ld = loss_data(model, xy_u_all, u_data_all, xy_v_all, v_data_all)
        lp = loss_physics_mct(
            model, xy_f_all,
            lambda_mct=1e-2,   # small compressibility constant
            n_iter=5,          # inner correction iterations
            tol=1e-6,
        )
        lb = loss_boundary(model, xy_b_all, u_bc_all)
        lg = loss_pressure_gauge(model, device)
        total = w_data * ld + w_phys_end * lp + w_bc * lb + w_gauge * lg
        total.backward()
        lbfgs_history["total"].append(total.item())
        lbfgs_history["data"].append(ld.item())
        lbfgs_history["physics"].append(lp.item())
        lbfgs_history["bc"].append(lb.item())
        lbfgs_history["gauge"].append(lg.item())
        lbfgs_history["Re"].append(model.reynolds_number().detach().item())
        return total

    for step in range(lbfgs_steps):
        optimizer_lbfgs.step(closure)
        if len(lbfgs_history["total"]) > 0:
            print(
                f"  L-BFGS step {step + 1:3d}/{lbfgs_steps} | "
                f"Total={lbfgs_history['total'][-1]:.3e} | "
                f"Data={lbfgs_history['data'][-1]:.3e} | "
                f"Phys={lbfgs_history['physics'][-1]:.3e} | "
                f"BC={lbfgs_history['bc'][-1]:.3e} | "
                f"Re={model.reynolds_number().detach().item():.3f}"
            )

    for k in history:
        history[k] += lbfgs_history[k]

    return history


# ============================================================
# 5) Evaluate on grid
# ============================================================

def evaluate_pinn_on_grid(model, x, y, device):
    Xg, Yg = np.meshgrid(x, y, indexing="xy")
    xy = np.column_stack([Xg.ravel(), Yg.ravel()])
    xy_t = torch.tensor(xy, dtype=torch.float32, device=device).requires_grad_(True)

    with torch.no_grad():
        u_p, v_p, p_p = model(xy_t)

    def to_grid(t):
        return t.detach().cpu().numpy().reshape(Xg.shape)

    return Xg, Yg, to_grid(u_p), to_grid(v_p), to_grid(p_p)

# ============================================================
# Execucao
# ============================================================

def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}\n")

    # Dados supervisionados: somente os perfis tabelados u e v.
    # Para usar os valores de "Ref. [12]", substitua por "referencia".
    dados = load_centerline_velocity_data(source="calculado")
    xy_u_all   = to_device_tensor(dados["xy_u"], device)
    u_data_all = to_device_tensor(dados["u"], device)
    xy_v_all   = to_device_tensor(dados["xy_v"], device)
    v_data_all = to_device_tensor(dados["v"], device)

    print(f"Pontos medidos de u: {len(u_data_all)} (x=0.5)")
    print(f"Pontos medidos de v: {len(v_data_all)} (y=0.5)")
    print("Nao sao usados dados medidos de pressao ou vorticidade.\n")

    # Condicoes do problema
    Lx, Ly = 1.0, 1.0
    U_lid = -1.0

    # Re e desconhecido e sera aprendido; 500 e apenas o chute inicial.
    Re_initial_guess = 500.0

    # Pontos de colocacao (residuos das EDPs) e de contorno:
    # continuam amostrados no dominio, independentemente das medidas.
    xy_f_all = make_sobol_points(2048, Lx, Ly, device, seed=188)
    xy_b_all, u_bc_all = make_boundary_points(256, Lx, Ly, U_lid, device, seed=88)

    model = CavityPINN_NS(
        hidden_layers=(128, 128, 128, 128),
        n_freq=48,
        sigma=4.0,
        Lx=Lx, Ly=Ly,
        U_ref=abs(U_lid),
        Re_init=Re_initial_guess,
    ).to(device)

    total_params = sum(param.numel() for param in model.parameters() if param.requires_grad)
    print(f"PINN (Navier-Stokes) parameters: {total_params:,}\n")

    print("Training PINN...")
    history = train_pinn(
        model=model,
        xy_u_all=xy_u_all,
        u_data_all=u_data_all,
        xy_v_all=xy_v_all,
        v_data_all=v_data_all,
        xy_f_all=xy_f_all,
        xy_b_all=xy_b_all,
        u_bc_all=u_bc_all,
        adam_epochs=4500,
        batch_f=512,
        batch_b=512,
        lr=1e-3,
        w_data=40.0,
        w_phys_start=1.0,
        w_phys_end=40.0,
        w_bc=40.0,
        w_gauge=10.0,
        lbfgs_steps=80,
        lbfgs_max_iter=200,
    )
    print("Training done.\n")

    Re_identified = model.reynolds_number().detach().cpu().item()
    nu_identified = model.kinematic_viscosity().detach().cpu().item()

    with torch.no_grad():
        u_pred = model(xy_u_all)[0]
        v_pred = model(xy_v_all)[1]
        rmse_u = torch.sqrt(torch.mean((u_pred - u_data_all)**2)).item()
        rmse_v = torch.sqrt(torch.mean((v_pred - v_data_all)**2)).item()

    print(f"Identified Reynolds number: Re = {Re_identified:.6f}")
    print(f"Corresponding kinematic viscosity: nu = {nu_identified:.6e}")
    print(f"RMSE u(x=0.5,y): {rmse_u:.6e}")
    print(f"RMSE v(x,y=0.5): {rmse_v:.6e}")

    # Comparacao entre as 34 observacoes e o modelo treinado.
    import matplotlib.pyplot as plt
    fig, axs = plt.subplots(1, 2, figsize=(11, 4))
    y_obs = dados["xy_u"][:, 1]
    x_obs = dados["xy_v"][:, 0]
    axs[0].plot(dados["u"].ravel(), y_obs, "o", label="Tabela u")
    axs[0].plot(u_pred.detach().cpu().numpy().ravel(), y_obs, "-", label="PINN")
    axs[0].set(xlabel="u", ylabel="y", title="Perfil u(x=0.5, y)")
    axs[1].plot(x_obs, dados["v"].ravel(), "o", label="Tabela v")
    axs[1].plot(x_obs, v_pred.detach().cpu().numpy().ravel(), "-", label="PINN")
    axs[1].set(xlabel="x", ylabel="v", title="Perfil v(x, y=0.5)")
    for ax in axs:
        ax.grid(alpha=0.3)
        ax.legend()
    fig.tight_layout()
    fig.savefig("PINN_perfis_tabelas.png", dpi=200)
    plt.close(fig)

    plt.figure(figsize=(7, 4))
    plt.plot(history["Re"])
    plt.xlabel("Atualizacoes registradas (Adam + avaliacoes L-BFGS)")
    plt.ylabel("Re estimado")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig("PINN_evolucao_Re.png", dpi=200)
    plt.close()

    return model, history, dados


if __name__ == "__main__":
    main()


# Com Pressão

In [ ]:
import torch
import torch.nn as nn
from torch.quasirandom import SobolEngine
import numpy as np

torch.manual_seed(88)
np.random.seed(88)

if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")

# ============================================================
# 2) Dados tabelados (u, v, p); sem omega
# ============================================================

def load_centerline_uvp_data(source="calculado"):
    """Le as duas tabelas de dados nos perfis centrais da cavidade.

    - 17 velocidades v(x, y=0.5)
    - 17 velocidades u(x=0.5, y)
    - 33 pressoes relativas p (17+17, sem repetir o centro)

    A rede tem 3 saidas (u, v, p). Omega NAO e usado.
    "calculado": usa as colunas v/u da simulacao tabelada (padrao)
    "referencia": usa as colunas Ref. [12] para u/v; ATENCAO:
        p e fornecido somente na coluna 'p' (nao ha p de referencia).
    Para comparar em condicoes consistentes, mantenha 'calculado'.

    Cada variavel tem seus pontos (x,y) de observacao especificos.
    Valores nao observados em um ponto nunca sao substituidos por zero.
    """
    # Tabela 1 - Perfil horizontal, velocidade vertical v em y=0.5
    x_v = np.array([
        0.0000, 0.0312, 0.0391, 0.0469, 0.0547, 0.0937,
        0.1406, 0.1953, 0.5000, 0.7656, 0.7734, 0.8437,
        0.9062, 0.9219, 0.9297, 0.9375, 1.0000,
    ], dtype=np.float32)

    v_calculado = np.array([
         0.0000000, -0.2279225, -0.2936869, -0.3553213,
        -0.4103754, -0.5264392, -0.4264545, -0.3202137,
         0.0257995,  0.3253592,  0.3339924,  0.3769189,
         0.3330442,  0.3099097,  0.2962703,  0.2807056,
         0.0000000,
    ], dtype=np.float32)

    v_referencia = np.array([
         0.00000, -0.21388, -0.27669, -0.33714, -0.39188,
        -0.51550, -0.42665, -0.31966,  0.02526,  0.32235,
         0.33075,  0.37095,  0.32627,  0.30353,  0.29012,
         0.27485,  0.00000,
    ], dtype=np.float32)

    # Pressao (relativa) na linha y=0.5, conforme a primeira tabela.
    p_v = np.array([
        0.077455, 0.078837, 0.078685, 0.078148, 0.077154,
        0.065816, 0.049029, 0.034552, 0.000000, 0.044848,
        0.047260, 0.069511, 0.084386, 0.086716, 0.087653,
        0.088445, 0.090477,
    ], dtype=np.float32)

    # Tabela 2 - Perfil vertical, velocidade horizontal u em x=0.5
    y_u = np.array([
        1.0000, 0.9766, 0.9688, 0.9609, 0.9531, 0.8516,
        0.7344, 0.6172, 0.5000, 0.4531, 0.2813, 0.1719,
        0.1016, 0.0703, 0.0625, 0.0547, 0.0000,
    ], dtype=np.float32)

    u_calculado = np.array([
        -1.0000000, -0.6644227, -0.5808359, -0.5169277,
        -0.4723329, -0.3372212, -0.1886747, -0.0570178,
         0.0620561,  0.1081999,  0.2803696,  0.3885691,
         0.3004561,  0.2228955,  0.2023300,  0.1812881,
         0.0000000,
    ], dtype=np.float32)

    u_referencia = np.array([
        -1.00000, -0.65928, -0.57492, -0.51117, -0.46604,
        -0.33304, -0.18719, -0.05702,  0.06080,  0.10648,
         0.27805,  0.38289,  0.29730,  0.22220,  0.20196,
         0.18109,  0.00000,
    ], dtype=np.float32)

    # Pressao (relativa) na linha x=0.5, conforme a segunda tabela.
    p_u = np.array([
        0.052987, 0.052009, 0.051514, 0.050949, 0.050329,
        0.034910, 0.012122, -0.000827, 0.000000, 0.004434,
        0.040377, 0.081925, 0.104187, 0.108566, 0.109200,
        0.109689, 0.110591,
    ], dtype=np.float32)

    if source == "calculado":
        u_obs, v_obs = u_calculado, v_calculado
    elif source == "referencia":
        u_obs, v_obs = u_referencia, v_referencia
    else:
        raise ValueError("source deve ser 'calculado' ou 'referencia'.")

    xy_u = np.column_stack((np.full_like(y_u, 0.5), y_u))
    xy_v = np.column_stack((x_v, np.full_like(x_v, 0.5)))

    assert len(xy_u) == len(u_obs) == len(p_u) == 17
    assert len(xy_v) == len(v_obs) == len(p_v) == 17
    assert p_u[8] == p_v[8] == 0.0  # Centro (0.5, 0.5)

    # O centro aparece em ambas as tabelas; retiramos a repeticao
    # apenas do conjunto de pressao, mantendo 33 pontos espaciais unicos.
    centro_v = np.where((xy_v[:, 0] == 0.5) & (xy_v[:, 1] == 0.5))[0]
    assert len(centro_v) == 1
    keep_v = np.ones(len(xy_v), dtype=bool)
    keep_v[centro_v[0]] = False
    xy_p = np.vstack([xy_u, xy_v[keep_v]])
    p_obs = np.concatenate([p_u, p_v[keep_v]])[:, None]
    assert len(xy_p) == 33

    return {
        "xy_u": xy_u, "u": u_obs[:, None],
        "xy_v": xy_v, "v": v_obs[:, None],
        "xy_p": xy_p, "p": p_obs,
        "p_u": p_u[:, None], "p_v": p_v[:, None],
        "u_ref": u_referencia[:, None], "v_ref": v_referencia[:, None],
    }


# ============================================================
# FIX 1 – Random Fourier Feature embedding
# ============================================================

class FourierEmbedding(nn.Module):
    """
    Maps (x, y) → [sin(Bx), cos(Bx)]  where B ~ N(0, σ²).
    Lets the network represent high-frequency spatial variations.
    """
    def __init__(self, in_dim: int = 2, n_freq: int = 32, sigma: float = 2.0):
        super().__init__()
        B = torch.randn(in_dim, n_freq) * sigma
        self.register_buffer("B", B)          # not a learnable param

    @property
    def out_dim(self):
        return 2 * self.B.shape[1]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        proj = x @ self.B                     # (N, n_freq)
        return torch.cat([torch.sin(proj),
                          torch.cos(proj)], dim=-1)   # (N, 2*n_freq)


def build_mlp(layer_sizes):
    modules = []
    for i in range(len(layer_sizes) - 2):
        modules += [nn.Linear(layer_sizes[i], layer_sizes[i + 1]), nn.Tanh()]
    modules.append(nn.Linear(layer_sizes[-2], layer_sizes[-1]))
    return nn.Sequential(*modules)


# ============================================================
# FIX 2 – Deeper network + Fourier input + input normalisation
# ============================================================

class CavityPINN_NS(nn.Module):
    """
    Outputs (u, v, p).
    """
    def __init__(
        self,
        hidden_layers = (128, 128, 128, 128),
        n_freq: int   = 32,
        sigma: float  = 2.0,
        Lx: float     = 1.0,
        Ly: float     = 1.0,
        U_ref: float  = 1.0,
        Re_init: float = 500.0,
    ):
        super().__init__()
        # Input normalisation buffers
        self.register_buffer("x_mean", torch.tensor([Lx / 2.0, Ly / 2.0]))
        self.register_buffer("x_std",  torch.tensor([Lx / 2.0, Ly / 2.0]))

        # Known reference scales used in Re = U_ref * L_ref / nu
        self.register_buffer("U_ref", torch.tensor(float(abs(U_ref))))
        self.register_buffer("L_ref", torch.tensor(float(Lx)))

        # Inverse parameter. Optimizing log(Re) guarantees Re > 0.
        if Re_init <= 0:
            raise ValueError("Re_init must be positive.")
        self.log_Re = nn.Parameter(
            torch.tensor(np.log(Re_init), dtype=torch.float32)
        )

        # Fourier embedding
        self.fourier = FourierEmbedding(2, n_freq, sigma)
        in_dim = self.fourier.out_dim

        # Output is now 3: u, v, p
        layer_sizes = [in_dim] + list(hidden_layers) + [3]
        self.net = build_mlp(layer_sizes)

    def reynolds_number(self):
        """Current positive Reynolds-number estimate."""
        return torch.exp(self.log_Re)

    def kinematic_viscosity(self):
        """nu = U_ref * L_ref / Re."""
        return self.U_ref * self.L_ref / self.reynolds_number()

    def forward(self, xy: torch.Tensor):
        xy_norm = (xy - self.x_mean) / self.x_std    # [-1, 1]
        feats   = self.fourier(xy_norm)
        out     = self.net(feats)
        return out[:, 0:1], out[:, 1:2], out[:, 2:3] # u, v, p


# ============================================================
# FIX 3 – Efficient gradient computation (Navier-Stokes)
# ============================================================

def physics_residuals(model, xy):
    """
    Continuity: u_x + v_y = 0
    X-Momentum: u*u_x + v*u_y + p_x - ν*(u_xx + u_yy) = 0
    Y-Momentum: u*v_x + v*v_y + p_y - ν*(v_xx + v_yy) = 0
    """
    # 1. Forward pass
    u, v, p = model(xy)

    # 2. First-order derivatives (Compute all at once for each output to save backward passes)
    u_grad = torch.autograd.grad(u, xy, grad_outputs=torch.ones_like(u), create_graph=True)[0]
    u_x, u_y = u_grad[:, 0:1], u_grad[:, 1:2]

    v_grad = torch.autograd.grad(v, xy, grad_outputs=torch.ones_like(v), create_graph=True)[0]
    v_x, v_y = v_grad[:, 0:1], v_grad[:, 1:2]

    p_grad = torch.autograd.grad(p, xy, grad_outputs=torch.ones_like(p), create_graph=True)[0]
    p_x, p_y = p_grad[:, 0:1], p_grad[:, 1:2]

    # 3. Second-order derivatives (Only need diagonals: u_xx, u_yy, v_xx, v_yy)
    u_x_grad = torch.autograd.grad(u_x, xy, grad_outputs=torch.ones_like(u_x), create_graph=True)[0]
    u_xx = u_x_grad[:, 0:1]
    
    u_y_grad = torch.autograd.grad(u_y, xy, grad_outputs=torch.ones_like(u_y), create_graph=True)[0]
    u_yy = u_y_grad[:, 1:2]

    v_x_grad = torch.autograd.grad(v_x, xy, grad_outputs=torch.ones_like(v_x), create_graph=True)[0]
    v_xx = v_x_grad[:, 0:1]

    v_y_grad = torch.autograd.grad(v_y, xy, grad_outputs=torch.ones_like(v_y), create_graph=True)[0]
    v_yy = v_y_grad[:, 1:2]

    # 4. Navier-Stokes Residuals
    # Viscosity is obtained from the trainable Reynolds number.
    nu = model.kinematic_viscosity()
    r_cont  = u_x + v_y
    r_mom_x = u * u_x + v * u_y + p_x - nu * (u_xx + u_yy)
    r_mom_y = u * v_x + v * v_y + p_y - nu * (v_xx + v_yy)

    return r_cont, r_mom_x, r_mom_y


# ============================================================
# 3) Collocation & boundary point generators
# ============================================================

def make_sobol_points(n_points, Lx, Ly, device, seed=123):
    sobol = SobolEngine(dimension=2, scramble=True, seed=seed)
    pts   = sobol.draw(n_points).to(device)
    pts[:, 0] *= Lx
    pts[:, 1] *= Ly
    return pts


def make_boundary_points(n_per_side, Lx, Ly, U_lid, device, seed=123):
    rng = np.random.default_rng(seed)

    xb = rng.random((n_per_side, 1)) * Lx;  yb = np.zeros((n_per_side, 1))
    xt = rng.random((n_per_side, 1)) * Lx;  yt = np.full((n_per_side, 1), Ly)
    xl = np.zeros((n_per_side, 1));          yl = rng.random((n_per_side, 1)) * Ly
    xr = np.full((n_per_side, 1), Lx);      yr = rng.random((n_per_side, 1)) * Ly

    xy_bc = np.vstack([
        np.hstack([xb, yb]), np.hstack([xt, yt]),
        np.hstack([xl, yl]), np.hstack([xr, yr]),
    ])
    u_bc = np.vstack([
        np.zeros((n_per_side, 1)),
        np.full((n_per_side, 1), U_lid),
        np.zeros((n_per_side, 1)),
        np.zeros((n_per_side, 1)),
    ])

    return (
        torch.tensor(xy_bc, dtype=torch.float32, device=device),
        torch.tensor(u_bc,  dtype=torch.float32, device=device),
    )

def to_device_tensor(arr, device):
    return torch.tensor(arr, dtype=torch.float32, device=device)


# ============================================================
# 4) Individual loss terms
# ============================================================

def loss_data(model, xy_u, u_obs, xy_v, v_obs,
              xy_p, p_obs, w_pressure_relative=3.0):
    """Perda supervisionada separada para u, v e p.

    A rede so e comparada com o campo que a tabela realmente informou:
      u(x=0.5, y), v(x, y=0.5), p nos dois perfis centrais.
    O peso relativo de pressao evita que seu MSE pequeno (p ~ 0.1)
    fique numericamente pouco influente diante de u/v (ate ~1).
    E um hiperparametro: nao garante uma estimativa especifica de Re.
    """
    u_pred, _, _ = model(xy_u)
    _, v_pred, _ = model(xy_v)
    _, _, p_pred = model(xy_p)
    loss_u = nn.functional.mse_loss(u_pred, u_obs)
    loss_v = nn.functional.mse_loss(v_pred, v_obs)
    loss_p = nn.functional.mse_loss(p_pred, p_obs)
    return loss_u + loss_v + w_pressure_relative * loss_p

# ============================================================
# Modified Compressibility Technique (MCT) for Pressure
# Based on: Dyn. eqs. with iterative pressure correction
#   p_correction = -lambda * D,  D = du/dx + dv/dy
#   p_new = p_old + p_correction
# ============================================================
def loss_pressure_gauge(model, device, x_ref=0.5, y_ref=0.5):
    """
    Pins the pressure at the cavity center to zero,
    removing the arbitrary constant in the pressure field.
    """
    xy_ref = torch.tensor([[x_ref, y_ref]], dtype=torch.float32, device=device)
    _, _, p_ref = model(xy_ref)
    return p_ref ** 2
def pressure_correction_step(model, xy, lambda_mct=1e-2, n_iter=5, tol=1e-6):
    """
    Applies the Modified Compressibility iterative pressure correction.

    Algorithm:
      1. Forward pass -> get u, v, p (initial p from network)
      2. Compute dilation D = u_x + v_y
      3. If |D| <= tol or iter >= limit: exit
      4. p_correction = -lambda * D
      5. p_new = p_old + p_correction
      6. Recompute gradients with updated p and repeat

    Returns corrected (u, v, p, r_cont, r_mom_x, r_mom_y)
    """
    xy = xy.detach().requires_grad_(True)

    # Initial forward pass
    u, v, p = model(xy)

    for _ in range(n_iter):
        # --- Compute dilation D = u_x + v_y ---
        u_grad = torch.autograd.grad(
            u, xy, grad_outputs=torch.ones_like(u), create_graph=True
        )[0]
        u_x, u_y = u_grad[:, 0:1], u_grad[:, 1:2]

        v_grad = torch.autograd.grad(
            v, xy, grad_outputs=torch.ones_like(v), create_graph=True
        )[0]
        v_x, v_y = v_grad[:, 0:1], v_grad[:, 1:2]

        D = u_x + v_y  # dilation / divergence of velocity

        # --- Check convergence ---
        if D.abs().mean().item() <= tol:
            break

        # --- Pressure correction: p_new = p_old - lambda * D ---
        p_correction = -lambda_mct * D
        p = p + p_correction.detach()   # detach correction to avoid graph issues

    # --- Final residuals with corrected pressure ---
    # Recompute all derivatives cleanly for residual loss
    xy_r = xy.detach().requires_grad_(True)
    u_r, v_r, _ = model(xy_r)
    # Use corrected pressure (detached from correction loop, kept differentiable via xy_r)
    _, _, p_r = model(xy_r)
    p_r = p_r + (-lambda_mct * (
        torch.autograd.grad(u_r, xy_r, grad_outputs=torch.ones_like(u_r), create_graph=True)[0][:, 0:1] +
        torch.autograd.grad(v_r, xy_r, grad_outputs=torch.ones_like(v_r), create_graph=True)[0][:, 1:2]
    )).detach()

    p_grad = torch.autograd.grad(
        p_r, xy_r, grad_outputs=torch.ones_like(p_r), create_graph=True
    )[0]
    p_x, p_y = p_grad[:, 0:1], p_grad[:, 1:2]

    u_grad2 = torch.autograd.grad(u_r, xy_r, grad_outputs=torch.ones_like(u_r), create_graph=True)[0]
    u_x2, u_y2 = u_grad2[:, 0:1], u_grad2[:, 1:2]

    v_grad2 = torch.autograd.grad(v_r, xy_r, grad_outputs=torch.ones_like(v_r), create_graph=True)[0]
    v_x2, v_y2 = v_grad2[:, 0:1], v_grad2[:, 1:2]

    u_xx = torch.autograd.grad(u_x2, xy_r, grad_outputs=torch.ones_like(u_x2), create_graph=True)[0][:, 0:1]
    u_yy = torch.autograd.grad(u_y2, xy_r, grad_outputs=torch.ones_like(u_y2), create_graph=True)[0][:, 1:2]
    v_xx = torch.autograd.grad(v_x2, xy_r, grad_outputs=torch.ones_like(v_x2), create_graph=True)[0][:, 0:1]
    v_yy = torch.autograd.grad(v_y2, xy_r, grad_outputs=torch.ones_like(v_y2), create_graph=True)[0][:, 1:2]

    # Reynolds number is inferred by the PINN; viscosity follows from it.
    nu = model.kinematic_viscosity()

    # Modified continuity: lambda * dp/dt + D = 0 (enforced as D -> 0)
    r_cont  = u_x2 + v_y2                                    # should -> 0 after correction
    r_mom_x = u_r * u_x2 + v_r * u_y2 + p_x - nu * (u_xx + u_yy)
    r_mom_y = u_r * v_x2 + v_r * v_y2 + p_y - nu * (v_xx + v_yy)

    return r_cont, r_mom_x, r_mom_y, D.detach()


def loss_physics_mct(model, xy_f, lambda_mct=1e-2, n_iter=5, tol=1e-6):
    """
    Physics loss using the Modified Compressibility Technique.
    Adds a dilation penalty term to explicitly drive D -> 0.
    """
    r_cont, r_mom_x, r_mom_y, D = pressure_correction_step(
        model, xy_f,
        lambda_mct=lambda_mct,
        n_iter=n_iter,
        tol=tol,
    )

    loss_continuity = torch.mean(r_cont ** 2)
    loss_momentum   = torch.mean(r_mom_x ** 2) + torch.mean(r_mom_y ** 2)
    loss_dilation   = torch.mean(D ** 2)   # explicit penalty on divergence

    return loss_continuity + loss_momentum + loss_dilation


def loss_boundary(model, xy_b, u_bc):
    """
    Explicit no-slip enforcement per wall:
      - Bottom (y=0) : u=0, v=0
      - Left   (x=0) : u=0, v=0
      - Right  (x=1) : u=0, v=0
      - Top    (y=1) : u=U_lid, v=0  (driven lid)
    """
    n = xy_b.shape[0] // 4   # points per side

    xy_bottom = xy_b[0*n : 1*n].detach().requires_grad_(True)
    xy_top    = xy_b[1*n : 2*n].detach().requires_grad_(True)
    xy_left   = xy_b[2*n : 3*n].detach().requires_grad_(True)
    xy_right  = xy_b[3*n : 4*n].detach().requires_grad_(True)

    u_lid_target = u_bc[1*n : 2*n]   # only the top has U_lid != 0

    # Predict fields
    u_bot, v_bot, _ = model(xy_bottom)
    u_top, v_top, _ = model(xy_top)
    u_lft, v_lft, _ = model(xy_left)
    u_rgt, v_rgt, _ = model(xy_right)

    # No-slip walls: u=0, v=0
    loss_no_slip = (
          torch.mean(u_bot**2) + torch.mean(v_bot**2)
        + torch.mean(u_lft**2) + torch.mean(v_lft**2)
        + torch.mean(u_rgt**2) + torch.mean(v_rgt**2)
    )

    # Driven lid: u=U_lid, v=0
    loss_lid = (
          torch.mean((u_top - u_lid_target)**2)
        + torch.mean(v_top**2)
    )

    return loss_no_slip + loss_lid

# ============================================================
# FIX 4+5 – Ramped physics weight + two-phase Adam → L-BFGS
# ============================================================

def train_pinn(
    model,
    xy_u_all, u_data_all, xy_v_all, v_data_all,
    xy_p_all, p_data_all,
    xy_f_all, xy_b_all, u_bc_all,
    adam_epochs  = 3000,
    batch_f      = 2048,
    batch_b      = 512,
    lr           = 1e-3,
    w_data       = 5.0,
    w_pressure_relative = 3.0,
    w_phys_start = 1.0,
    w_phys_end   = 10.0,
    w_bc         = 10.0,
    w_gauge        = 10.0,
    lbfgs_steps  = 60,
    lbfgs_max_iter = 200,
):
    device = xy_f_all.device
    Nf, Nb = xy_f_all.shape[0], xy_b_all.shape[0]

    # ---- Adam phase ----
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.MultiStepLR(
        optimizer, milestones=[1000, 2000], gamma=0.3
    )

    history = {"total": [], "data": [], "physics": [], "bc": [], "gauge": [], "Re": []}

    for epoch in range(adam_epochs):
        idx_f = torch.randint(0, Nf, (min(batch_f,  Nf),),  device=device)

        t = epoch / max(adam_epochs - 1, 1)
        w_phys = w_phys_start + t * (w_phys_end - w_phys_start)

        optimizer.zero_grad()

        # Todos os dados (17 u, 17 v, 33 p) entram em toda epoca.
        ld = loss_data(model, xy_u_all, u_data_all, xy_v_all, v_data_all,
                       xy_p_all, p_data_all, w_pressure_relative)
        lp = loss_physics_mct(
            model, xy_f_all[idx_f],
            lambda_mct=1e-2,   # small compressibility constant
            n_iter=5,          # inner correction iterations
            tol=1e-6,
        )

        # Sample per wall to keep ordering intact:
        n_wall = Nb // 4
        b_per_wall = min(batch_b // 4, n_wall)
        idx_b = torch.cat([
            torch.randint(i*n_wall, (i+1)*n_wall, (b_per_wall,), device=device)
            for i in range(4)
        ])
        lb = loss_boundary(model, xy_b_all[idx_b], u_bc_all[idx_b])

        lg = loss_pressure_gauge(model, device)

        loss = w_data * ld + w_phys * lp + w_bc * lb + w_gauge * lg
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        history["total"].append(loss.item())
        history["data"].append(ld.item())
        history["physics"].append(lp.item())
        history["bc"].append(lb.item())

        history["gauge"].append(lg.item())
        history["Re"].append(model.reynolds_number().detach().item())

        if epoch % 300 == 0:
            print(
                f"  Adam {epoch:5d}/{adam_epochs} | "
                f"Total={loss.item():.3e} | "
                f"Data={ld.item():.3e} | "
                f"Phys={lp.item():.3e} (w={w_phys:.1f}) | "
                f"BC={lb.item():.3e} | "
                f"Gauge={lg.item():.3e} | "
                f"Re={model.reynolds_number().detach().item():.3f}"
            )

    # ---- FIX 5: L-BFGS phase (full-batch) ----
    print("\nStarting L-BFGS phase...")
    optimizer_lbfgs = torch.optim.LBFGS(
        model.parameters(),
        lr=0.1,
        max_iter=lbfgs_max_iter,
        tolerance_grad=1e-9,
        tolerance_change=1e-12,
        history_size=100,
        line_search_fn="strong_wolfe",
    )

    lbfgs_history = {"total": [], "data": [], "physics": [], "bc": [], "gauge": [], "Re": []}

    def closure():
        optimizer_lbfgs.zero_grad()
        ld = loss_data(model, xy_u_all, u_data_all, xy_v_all, v_data_all,
                       xy_p_all, p_data_all, w_pressure_relative)
        lp = loss_physics_mct(
            model, xy_f_all,
            lambda_mct=1e-2,   # small compressibility constant
            n_iter=5,          # inner correction iterations
            tol=1e-6,
        )
        lb = loss_boundary(model, xy_b_all, u_bc_all)
        lg = loss_pressure_gauge(model, device)
        total = w_data * ld + w_phys_end * lp + w_bc * lb + w_gauge * lg
        total.backward()
        lbfgs_history["total"].append(total.item())
        lbfgs_history["data"].append(ld.item())
        lbfgs_history["physics"].append(lp.item())
        lbfgs_history["bc"].append(lb.item())
        lbfgs_history["gauge"].append(lg.item())
        lbfgs_history["Re"].append(model.reynolds_number().detach().item())
        return total

    for step in range(lbfgs_steps):
        optimizer_lbfgs.step(closure)
        if len(lbfgs_history["total"]) > 0:
            print(
                f"  L-BFGS step {step + 1:3d}/{lbfgs_steps} | "
                f"Total={lbfgs_history['total'][-1]:.3e} | "
                f"Data={lbfgs_history['data'][-1]:.3e} | "
                f"Phys={lbfgs_history['physics'][-1]:.3e} | "
                f"BC={lbfgs_history['bc'][-1]:.3e} | "
                f"Re={model.reynolds_number().detach().item():.3f}"
            )

    for k in history:
        history[k] += lbfgs_history[k]

    return history


# ============================================================
# 5) Evaluate on grid
# ============================================================

def evaluate_pinn_on_grid(model, x, y, device):
    Xg, Yg = np.meshgrid(x, y, indexing="xy")
    xy = np.column_stack([Xg.ravel(), Yg.ravel()])
    xy_t = torch.tensor(xy, dtype=torch.float32, device=device).requires_grad_(True)

    with torch.no_grad():
        u_p, v_p, p_p = model(xy_t)

    def to_grid(t):
        return t.detach().cpu().numpy().reshape(Xg.shape)

    return Xg, Yg, to_grid(u_p), to_grid(v_p), to_grid(p_p)

# ============================================================
# Execucao
# ============================================================

def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}\n")

    # Dados supervisionados: velocidades E pressao das duas tabelas.
    # Colunas u/v (nao as Ref. [12]) combinam com os p publicados.
    dados = load_centerline_uvp_data(source="calculado")
    xy_u_all   = to_device_tensor(dados["xy_u"], device)
    u_data_all = to_device_tensor(dados["u"], device)
    xy_v_all   = to_device_tensor(dados["xy_v"], device)
    v_data_all = to_device_tensor(dados["v"], device)
    xy_p_all   = to_device_tensor(dados["xy_p"], device)
    p_data_all = to_device_tensor(dados["p"], device)

    print(f"Observacoes u: {len(u_data_all)} (x=0.5)")
    print(f"Observacoes v: {len(v_data_all)} (y=0.5)")
    print(f"Observacoes p: {len(p_data_all)} (2 linhas; centro unico)")
    print("Vorticidade omega nao utilizada. Nao ha CFD previo.\n")

    # Condicoes do problema
    Lx, Ly = 1.0, 1.0
    U_lid = -1.0

    # Re e desconhecido e sera aprendido; 500 e apenas o chute inicial.
    Re_initial_guess = 500.0

    # Pontos de colocacao (residuos das EDPs) e de contorno:
    # continuam amostrados no dominio, independentemente das medidas.
    xy_f_all = make_sobol_points(2048, Lx, Ly, device, seed=188)
    xy_b_all, u_bc_all = make_boundary_points(256, Lx, Ly, U_lid, device, seed=88)

    model = CavityPINN_NS(
        hidden_layers=(128, 128, 128, 128),
        n_freq=48,
        sigma=4.0,
        Lx=Lx, Ly=Ly,
        U_ref=abs(U_lid),
        Re_init=Re_initial_guess,
    ).to(device)

    total_params = sum(param.numel() for param in model.parameters() if param.requires_grad)
    print(f"PINN (Navier-Stokes) parameters: {total_params:,}\n")

    print("Training PINN...")
    history = train_pinn(
        model=model,
        xy_u_all=xy_u_all,
        u_data_all=u_data_all,
        xy_v_all=xy_v_all,
        v_data_all=v_data_all,
        xy_p_all=xy_p_all,
        p_data_all=p_data_all,
        xy_f_all=xy_f_all,
        xy_b_all=xy_b_all,
        u_bc_all=u_bc_all,
        adam_epochs=4500,
        batch_f=512,
        batch_b=512,
        lr=1e-3,
        w_data=40.0,
        w_pressure_relative=3.0,  # peso relativo do termo supervisionado p
        w_phys_start=1.0,
        w_phys_end=40.0,
        w_bc=40.0,
        w_gauge=10.0,
        lbfgs_steps=80,
        lbfgs_max_iter=200,
    )
    print("Training done.\n")

    Re_identified = model.reynolds_number().detach().cpu().item()
    nu_identified = model.kinematic_viscosity().detach().cpu().item()

    with torch.no_grad():
        u_pred = model(xy_u_all)[0]
        v_pred = model(xy_v_all)[1]
        p_pred = model(xy_p_all)[2]
        rmse_u = torch.sqrt(torch.mean((u_pred - u_data_all)**2)).item()
        rmse_v = torch.sqrt(torch.mean((v_pred - v_data_all)**2)).item()
        rmse_p = torch.sqrt(torch.mean((p_pred - p_data_all)**2)).item()
        # As colunas 'Ref. [12]' sao comparacao independente,
        # NAO representam medidas extras usadas no treinamento.
        rmse_u_ref = torch.sqrt(torch.mean(
            (u_pred - to_device_tensor(dados["u_ref"], device))**2)).item()
        rmse_v_ref = torch.sqrt(torch.mean(
            (v_pred - to_device_tensor(dados["v_ref"], device))**2)).item()

    print(f"Identified Reynolds number: Re = {Re_identified:.6f}")
    print(f"Corresponding kinematic viscosity: nu = {nu_identified:.6e}")
    print(f"RMSE u(x=0.5,y): {rmse_u:.6e}")
    print(f"RMSE v(x,y=0.5): {rmse_v:.6e}")
    print(f"RMSE p(2 perfis): {rmse_p:.6e}")
    print(f"RMSE u comparado a Ref. [12]: {rmse_u_ref:.6e}")
    print(f"RMSE v comparado a Ref. [12]: {rmse_v_ref:.6e}")

    # Graficos de cada variavel observada versus PINN.
    import matplotlib.pyplot as plt
    fig, axs = plt.subplots(2, 2, figsize=(12, 9))
    y_obs = dados["xy_u"][:, 1]
    x_obs = dados["xy_v"][:, 0]
    u_np = u_pred.detach().cpu().numpy().ravel()
    v_np = v_pred.detach().cpu().numpy().ravel()
    p_np = p_pred.detach().cpu().numpy().ravel()
    p_u_pred = p_np[:17]
    # O centro da linha horizontal foi removido apenas em p_obs,
    # portanto para os graficos avaliamos p na linha horizontal completa.
    with torch.no_grad():
        p_v_pred = model(xy_v_all)[2].detach().cpu().numpy().ravel()

    axs[0, 0].plot(dados["u"].ravel(), y_obs, "o", label="Tabela u")
    axs[0, 0].plot(u_np, y_obs, "-", label="PINN")
    axs[0, 0].plot(dados["u_ref"].ravel(), y_obs, "--", label="Ref. [12]")
    axs[0, 0].set(xlabel="u", ylabel="y", title="Velocidade u (x=0.5)")

    axs[0, 1].plot(x_obs, dados["v"].ravel(), "o", label="Tabela v")
    axs[0, 1].plot(x_obs, v_np, "-", label="PINN")
    axs[0, 1].plot(x_obs, dados["v_ref"].ravel(), "--", label="Ref. [12]")
    axs[0, 1].set(xlabel="x", ylabel="v", title="Velocidade v (y=0.5)")

    axs[1, 0].plot(dados["p_u"].ravel(), y_obs, "o", label="Tabela p")
    axs[1, 0].plot(p_u_pred, y_obs, "-", label="PINN")
    axs[1, 0].set(xlabel="p", ylabel="y", title="Pressao (x=0.5)")

    axs[1, 1].plot(x_obs, dados["p_v"].ravel(), "o", label="Tabela p")
    axs[1, 1].plot(x_obs, p_v_pred, "-", label="PINN")
    axs[1, 1].set(xlabel="x", ylabel="p", title="Pressao (y=0.5)")

    for ax in axs.flat:
        ax.grid(alpha=0.3)
        ax.legend()
    fig.tight_layout()
    fig.savefig("PINN_perfis_uvp_tabelas.png", dpi=200)
    plt.close(fig)

    # Linhas de corrente previstas SOMENTE pela PINN.
    x_vis = np.linspace(0.0, Lx, 201)
    y_vis = np.linspace(0.0, Ly, 201)
    _, _, U_vis, V_vis, _ = evaluate_pinn_on_grid(model, x_vis, y_vis, device)
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.streamplot(x_vis, y_vis, U_vis, V_vis,
                  color="black", density=2.0, linewidth=0.8, arrowsize=0.8)
    ax.set(xlabel="x", ylabel="y", title="Linhas de corrente - PINN",
           xlim=(0, Lx), ylim=(0, Ly))
    ax.set_aspect("equal")
    fig.tight_layout()
    fig.savefig("PINN_linhas_corrente.png", dpi=200)
    plt.close(fig)

    plt.figure(figsize=(7, 4))
    plt.plot(history["Re"])
    plt.xlabel("Atualizacoes registradas (Adam + avaliacoes L-BFGS)")
    plt.ylabel("Re estimado")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig("PINN_evolucao_Re.png", dpi=200)
    plt.close()

    # Arquivo util para reutilizar o modelo sem treinar novamente.
    torch.save({"state_dict": model.state_dict(),
                "Re_identificado": Re_identified,
                "RMSE_u": rmse_u, "RMSE_v": rmse_v, "RMSE_p": rmse_p,
                "RMSE_u_ref": rmse_u_ref, "RMSE_v_ref": rmse_v_ref},
               "PINN_uvp_tabelas_modelo.pth")

    return model, history, dados


if __name__ == "__main__":
    main()
